# Quantitative Brain MRI Enhances Clinical Stratification of Hepatic Encephalopathy Beyond MELD: A Multicenter Study

## Pipeline Overview

This notebook implements the full statistical and machine learning pipeline described in the manuscript, in compliance with **TRIPOD** and **TRIPOD-AI** reporting guidelines (Collins et al., *Ann Intern Med* 2015; *BMJ* 2024).

### Study Design
- **Population:** 168 adults with cirrhosis, retrospective multicenter cohort (2017–2024)
- **Index test:** Radiomic features from T1-weighted MRI of the globus pallidus (PyRadiomics v3.0, IBSI-compliant)
- **Reference standard:** HE severity by West Haven criteria (graded independently before MRI analysis)
- **Clinical predictor:** MELD score at time of MRI

### Binary Outcomes
| Outcome | Task | Positive class |
|---|---|---|
| **Outcome 1** | Mild HE vs No HE | GRADE_HE = 1 vs 0 |
| **Outcome 2** | Moderate-to-Severe HE vs Mild HE | GRADE_HE ≥ 2 vs 1 |

### Models per Outcome
| Model | Predictors |
|---|---|
| **MELD** | MELD score only |
| **Radiomics** | LASSO-selected radiomic features |
| **Combined** | MELD + LASSO-selected radiomic features |

### Pipeline Structure
| Cell | Content |
|---|---|
| **1** | Setup — imports, paths, constants, palette |
| **2** | Data loading, integrity checks, decimal correction, cleaning |
| **3** | Subset creation for Outcome 1 and Outcome 2 with deterministic indexing |
| **4** | Nested 5-fold CV (outer: performance, inner: hyperparameter tuning) — OOF predictions |
| **5** | Performance metrics: AUC + 95% bootstrap CI, Brier score, calibration slope/intercept, DeLong test |
| **6** | Calibration plots (reliability diagrams) |
| **7** | Decision Curve Analysis (DCA) — net benefit across threshold probabilities |
| **8** | SHAP feature importance (LinearExplainer, top-20 features) |
| **9** | AUC grouped bar chart with DeLong significance annotations |
| **10** | Cross-system validation (Signa 1.5T ↔ Discovery 3.0T) |
| **11** | Clinical nomograms (MELD + RadScore) |
| **12** | Reproducibility check + session info |

### Key Methodological Notes
- **No data leakage:** All preprocessing (scaling, feature selection) is performed inside each training fold
- **OOF predictions:** One predicted probability per subject from the outer CV loop — used for all downstream metrics
- **RadScore:** Logit of OOF radiomics probability — leakage-free, used in nomograms
- **Bootstrap CI:** n = 2,000 resamples (percentile method)
- **DeLong test:** Two-tailed, paired comparison vs MELD reference
- **Cross-system:** Leave-one-system-out, 2,000-bootstrap AUC CI + 2,000-iteration permutation test

### TRIPOD-AI Compliance
> Collins GS, Moons KGM, Dhiman P, et al. TRIPOD+AI statement.  
> *BMJ* 2024;385:e078378. https://doi.org/10.1136/bmj-2023-078378

---

## License

This code is released under the **MIT License**.
```
MIT License

Copyright (c) [Year] [Authors]

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in
all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN
THE SOFTWARE.
```

---

## Code Availability

The full analysis code is available at:

**GitHub repository:** [ ]

Following publication, the code and dataset description will be permanently archived on **Zenodo** and assigned a citable DOI to ensure long-term, stable access in accordance with open-science and reproducibility standards.

---

## How to Cite

If you use this code in your research, please cite the companion manuscript:

> [Authors]. Quantitative Brain MRI Enhances Clinical Stratification of Hepatic Encephalopathy Beyond MELD: A Multicenter Study. DOI: [ ]

**BibTeX:**
```bibtex
@article{HE_Radiomics,
  author  = {[Authors]},
  title   = {Quantitative Brain MRI Enhances Clinical Stratification of
             Hepatic Encephalopathy Beyond MELD: A Multicenter Study},
  journal = {Hepatology},
  year    = {[Year]},
  doi     = {[DOI]},
}
```

If citing the code archive directly (post-publication):

> [Authors]. Analysis code for: Quantitative Brain MRI Enhances Clinical Stratification of Hepatic Encephalopathy Beyond MELD. Zenodo [Year]. DOI: [ ]

**BibTeX:**
```bibtex
@software{HE_Radiomics_Code,
  author    = {[Authors]},
  title     = {Analysis code for: Quantitative Brain MRI Enhances Clinical
               Stratification of Hepatic Encephalopathy Beyond MELD},
  publisher = {Zenodo},
  year      = {[Year]},
  doi       = {[Zenodo DOI]},
}
```
## Disclaimer

This tool is intended for **research purposes only**.  
It has not been validated for clinical decision-making and should not  
be used as a substitute for professional medical judgment.

---

*Last updated: [Year] — radiomics_pipeline.ipynb*

In [ ]:
# =============================================================================
# Cell 1 — Setup: imports, paths, random state, palette, constants
# =============================================================================
# TRIPOD-AI compliant pipeline for hepatic encephalopathy severity
# stratification using MRI-derived radiomic features and MELD score.
#
# Outcomes:
#   Outcome 1 — Mild HE vs No HE              (GRADE_HE: 1 vs 0)
#   Outcome 2 — Moderate-to-Severe vs Mild HE  (GRADE_HE: ≥2 vs 1)
#
# Models evaluated per outcome:
#   (i)   MELD only
#   (ii)  Radiomics only   — LASSO-selected IBSI-compliant features
#   (iii) Combined         — MELD + LASSO-selected radiomic features
#
# Authors  : [Authors]
# Journal  : Hepatology
# Python   : >= 3.9
# License  : MIT
# =============================================================================

# --- Standard library ---
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# --- Numerical & data ---
import numpy as np
import pandas as pd

# --- Machine learning ---
from sklearn.pipeline        import Pipeline
from sklearn.preprocessing   import StandardScaler
from sklearn.linear_model    import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics         import roc_auc_score, brier_score_loss
from sklearn.utils           import resample

# --- Statistics ---
import scipy.stats as stats

# --- Explainability ---
import shap

# --- KDE (nomograms) ---
from scipy.stats import gaussian_kde

# --- Plotting ---
import matplotlib
import matplotlib.pyplot  as plt
import matplotlib.patches as mpatches
matplotlib.rcParams.update({
    "font.family":       "sans-serif",
    "font.sans-serif":   ["Arial", "Helvetica"],
    "pdf.fonttype":      42,    # editable text in vector graphics
    "font.size":         11,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.linewidth":    1.0,
})

# --- Reporting ---
from IPython.display import display, HTML

# =============================================================================
# REPRODUCIBILITY
# =============================================================================
RANDOM_STATE = 42
BOOT_SEED    = 42
np.random.seed(RANDOM_STATE)

# =============================================================================
# PROJECT PATHS
# =============================================================================
ROOT     = Path(".")
DATA_DIR = ROOT / "data"
OUT_DIR  = ROOT / "outputs"
FIG_DIR  = OUT_DIR / "figures"
TAB_DIR  = OUT_DIR / "tables"
NOM_DIR  = FIG_DIR / "nomograms"
SHA_DIR  = FIG_DIR / "shap"
CAL_DIR  = FIG_DIR / "calibration"
DCA_DIR  = FIG_DIR / "dca"
BAR_DIR  = FIG_DIR / "barcharts"
CRS_DIR  = FIG_DIR / "crosssystem"

for d in [FIG_DIR, TAB_DIR, NOM_DIR, SHA_DIR,
          CAL_DIR, DCA_DIR, BAR_DIR, CRS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Input file — edit path if your CSV is named differently
RAW_CSV = DATA_DIR / "radiomic_HE_dataset.csv"

# =============================================================================
# MODEL HYPERPARAMETERS
# =============================================================================
CV_FOLDS = 5                                   # outer & inner CV folds
C_GRID   = {"clf__C": np.logspace(-3, 0, 10)}  # L1 regularisation grid
N_BOOT   = 2000                                # bootstrap resamples for CI

# =============================================================================
# COLUMN NAME CONSTANTS
# Edit these if your CSV uses different column names.
# =============================================================================
COL_MR_SYSTEM = "MR_System"        # scanner identifier (Signa / Discovery)
COL_MELD      = "MELD"
COL_GRADE     = "GRADE_HE"         # 0 = no HE, 1 = mild, ≥2 = mod-severe
COL_HE_NONE   = "HE_None"          # binary: 1 if GRADE_HE == 0
COL_HE_MILD   = "HE_Mild"          # binary: 1 if GRADE_HE == 1
COL_HE_MODSEV = "HE_ModerateSevere"# binary: 1 if GRADE_HE >= 2

# =============================================================================
# COLOUR PALETTE
# Standard colours — identical for both outcomes:
#   MELD      → green  #079A02
#   Radiomics → blue   #005DCE
#   Combined  → red    #A60628
# SHAP retains its own diverging palette (not overridden here).
# =============================================================================
MODEL_COLORS = {
    "MELD":      "#079A02",
    "Radiomics": "#005DCE",
    "Combined":  "#A60628",
}
MODEL_LABELS = ["MELD", "Radiomics", "Combined"]

# =============================================================================
# SCANNER LABELS (cross-system validation)
# =============================================================================
SCANNER_A   = "Signa"      # 1.5T
SCANNER_B   = "Discovery"  # 3.0T
CROSS_PATHS = [
    (SCANNER_A, SCANNER_B, "1.5T → 3.0T"),
    (SCANNER_B, SCANNER_A, "3.0T → 1.5T"),
]

# =============================================================================
# SUMMARY
# =============================================================================
print("✅  Cell 1 — Setup complete.")
print(f"    Output root  : {OUT_DIR.resolve()}")
print(f"    Random state : {RANDOM_STATE}")
print(f"    CV folds     : {CV_FOLDS}")
print(f"    C grid       : {C_GRID['clf__C'].round(4)}")
print(f"    Bootstrap n  : {N_BOOT}")
print(f"    Scanners     : {SCANNER_A} (1.5T)  |  {SCANNER_B} (3.0T)")

In [ ]:
# =============================================================================
# Cell 2 — Data loading, integrity checks, preprocessing, and cleaning
# =============================================================================
# Loads the raw radiomic + clinical dataset and performs the following steps:
#
# 1. Decimal correction  : converts European comma separators to periods
#                          to ensure numeric parsing across locales.
# 2. Scanner mapping     : maps MR_System string labels to field-strength
#                          tags (1.5T / 3.0T) for cross-system validation.
# 3. Integrity checks    : verifies that all required columns are present,
#                          that binary outcome columns are consistent with
#                          GRADE_HE (source of truth), and that radiomic
#                          feature columns follow the expected prefix
#                          convention ("original_").
# 4. Missing data        : rows with any missing value in predictors or
#                          outcomes are removed and logged. Per the
#                          manuscript, no missing data were present among
#                          the final 168 patients after applying predefined
#                          exclusion criteria.
# 5. Clean dataset saved : outputs/tables/radiomic_HE_dataset_clean.csv
#                          for downstream reproducibility (TRIPOD-AI item 15).
#
# TRIPOD-AI items covered:
#   - 4b : data source description
#   - 5c : outcome definition and source
#   - 7a : handling of missing data
#   - 15  : reproducibility — clean dataset saved to disk
# =============================================================================

# =============================================================================
# STEP 1 — Load raw CSV
# =============================================================================
if not RAW_CSV.exists():
    raise FileNotFoundError(
        f"Raw dataset not found: {RAW_CSV}\n"
        f"Please place your CSV at: {RAW_CSV.resolve()}"
    )

df_raw = pd.read_csv(RAW_CSV, sep=None, engine="python", dtype=str)
print(f"  Loaded  : {RAW_CSV.name}")
print(f"  Shape   : {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")

# =============================================================================
# STEP 2 — Decimal correction (European locale: comma → period)
# =============================================================================
def fix_decimals(df):
    """
    Replace comma decimal separators with periods in all columns,
    then coerce to numeric where possible.
    String-only columns (e.g. MR_System) are left unchanged.
    """
    df = df.copy()
    for col in df.columns:
        converted = (
            df[col]
            .str.replace(",", ".", regex=False)
            .pipe(pd.to_numeric, errors="coerce")
        )
        # Only apply if at least one value was successfully converted
        if converted.notna().any():
            df[col] = converted
    return df

df = fix_decimals(df_raw)
print(f"\n  ✓ Decimal correction applied.")

# =============================================================================
# STEP 3 — Scanner mapping
# Maps MR_System string to a standardised field-strength label.
# Edit the dictionary below if your CSV uses different scanner names.
# =============================================================================
SCANNER_MAP = {
    SCANNER_A: "1.5T",   # e.g. "Signa"     → "1.5T"
    SCANNER_B: "3.0T",   # e.g. "Discovery" → "3.0T"
}

if COL_MR_SYSTEM in df.columns:
    df[COL_MR_SYSTEM] = df_raw[COL_MR_SYSTEM].map(SCANNER_MAP)
    unmapped = df[COL_MR_SYSTEM].isna().sum()
    if unmapped > 0:
        print(f"\n  ⚠️  {unmapped} row(s) with unrecognised MR_System value.")
    else:
        print(f"  ✓ Scanner mapping complete: {df[COL_MR_SYSTEM].value_counts().to_dict()}")
else:
    print(f"\n  ⚠️  Column '{COL_MR_SYSTEM}' not found — cross-system validation will be skipped.")

# =============================================================================
# STEP 4 — Required columns check
# =============================================================================
REQUIRED_COLS = [COL_MELD, COL_GRADE, COL_HE_NONE, COL_HE_MILD, COL_HE_MODSEV]
missing_cols  = [c for c in REQUIRED_COLS if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required column(s): {missing_cols}")
print(f"  ✓ All required columns present.")

# Radiomic feature columns (IBSI-compliant prefix "original_")
rad_cols = [c for c in df.columns if c.startswith("original_")]
if len(rad_cols) == 0:
    raise ValueError(
        "No radiomic feature columns found (expected prefix 'original_').\n"
        "Check your CSV column names."
    )
print(f"  ✓ Radiomic features found: {len(rad_cols)}")

# =============================================================================
# STEP 5 — Outcome integrity check
# Binary outcome columns must be consistent with GRADE_HE (source of truth).
# =============================================================================
def check_outcome_consistency(df):
    """
    Verify that the three binary outcome columns are derivable from GRADE_HE.
    Logs any inconsistencies without raising an error — the GRADE_HE column
    is used to re-derive outcomes if discrepancies are found.
    """
    expected_none   = (df[COL_GRADE] == 0).astype(int)
    expected_mild   = (df[COL_GRADE] == 1).astype(int)
    expected_modsev = (df[COL_GRADE] >= 2).astype(int)

    for col, expected in zip(
        [COL_HE_NONE, COL_HE_MILD, COL_HE_MODSEV],
        [expected_none, expected_mild, expected_modsev],
    ):
        mismatch = (df[col].astype(int) != expected).sum()
        if mismatch > 0:
            print(f"  ⚠️  {mismatch} inconsistency(ies) in '{col}' — "
                  f"re-deriving from {COL_GRADE}.")
            df[col] = expected
        else:
            print(f"  ✓ '{col}' consistent with {COL_GRADE}.")
    return df

df = check_outcome_consistency(df)

# =============================================================================
# STEP 6 — GRADE_HE distribution
# =============================================================================
grade_counts = df[COL_GRADE].value_counts().sort_index()
print(f"\n  GRADE_HE distribution:")
labels_map = {0: "Grade 0 — No HE", 1: "Grade 1 — Mild HE", 2: "Grade ≥2 — Mod-Severe HE"}
for grade, count in grade_counts.items():
    label = labels_map.get(int(grade), f"Grade {int(grade)}")
    pct   = 100 * count / len(df)
    print(f"    {label} : {count} ({pct:.1f}%)")

# =============================================================================
# STEP 7 — Missing data
# Per the manuscript, no missing data are expected in the final cohort.
# Any rows with missing predictor or outcome values are flagged and removed.
# =============================================================================
predictor_cols = [COL_MELD] + rad_cols
outcome_cols   = [COL_GRADE, COL_HE_NONE, COL_HE_MILD, COL_HE_MODSEV]
check_cols     = predictor_cols + outcome_cols

n_before = len(df)
df        = df.dropna(subset=check_cols).reset_index(drop=True)
n_removed = n_before - len(df)

if n_removed > 0:
    print(f"\n  ⚠️  {n_removed} row(s) removed due to missing values.")
else:
    print(f"\n  ✓ No missing values in predictors or outcomes.")
print(f"  Final cohort: {len(df)} patients.")

# =============================================================================
# STEP 8 — Save clean dataset (TRIPOD-AI item 15)
# =============================================================================
CLEAN_CSV = TAB_DIR / "radiomic_HE_dataset_clean.csv"
df.to_csv(CLEAN_CSV, index=False)
print(f"\n  ✓ Clean dataset saved → {CLEAN_CSV.resolve()}")

# =============================================================================
# SUMMARY
# =============================================================================
print(f"\n✅  Cell 2 — Data loading and cleaning complete.")
print(f"    Patients       : {len(df)}")
print(f"    Radiomic cols  : {len(rad_cols)}")
print(f"    Scanner col    : {COL_MR_SYSTEM in df.columns}")
print(f"    Clean CSV      : {CLEAN_CSV.name}")

In [ ]:
# =============================================================================
# Cell 3 — Subset creation for Outcome 1 and Outcome 2
# =============================================================================
# Creates the two analysis subsets from the clean dataset.
#
# Outcome 1 — Mild HE vs No HE:
#   Patients with GRADE_HE = 0 (no HE) or GRADE_HE = 1 (mild HE).
#   Binary label: HE_Mild (1 = mild HE, 0 = no HE).
#
# Outcome 2 — Moderate-to-Severe HE vs Mild HE:
#   Patients with GRADE_HE = 1 (mild HE) or GRADE_HE >= 2 (mod-severe HE).
#   Binary label: HE_ModerateSevere (1 = mod-severe, 0 = mild HE).
#
# CRITICAL — reset_index(drop=True):
#   Applied to all subsets to guarantee that StratifiedKFold with
#   shuffle=True and fixed random_state produces identical fold
#   assignments across independent runs. Without this, residual
#   integer indices from the parent dataframe can alter fold boundaries.
#
# CRITICAL — feature column freeze:
#   rad_cols is defined once here from the clean dataset and reused
#   in all downstream cells. This prevents silent column drift if the
#   dataframe is modified later.
#
# TRIPOD-AI items covered:
#   - 5b  : outcome definition and patient selection per task
#   - 5c  : predictors and their measurement
#   - 8a  : sample size per outcome
# =============================================================================

# =============================================================================
# STEP 1 — Outcome 1: Mild HE vs No HE
#           GRADE_HE ∈ {0, 1}  |  label = HE_Mild
# =============================================================================
mask_o1 = df[COL_GRADE].isin([0, 1])
df_o1   = df[mask_o1].reset_index(drop=True)
y_o1    = df_o1[COL_HE_MILD].astype(int).values

n_o1       = len(df_o1)
n_o1_pos   = int(y_o1.sum())          # mild HE (positive class)
n_o1_neg   = n_o1 - n_o1_pos          # no HE   (negative class)
prev_o1    = n_o1_pos / n_o1

print("  Outcome 1 — Mild HE vs No HE")
print(f"    Total patients : {n_o1}")
print(f"    Mild HE  (y=1) : {n_o1_pos} ({100*prev_o1:.1f}%)")
print(f"    No HE    (y=0) : {n_o1_neg} ({100*(1-prev_o1):.1f}%)")

# =============================================================================
# STEP 2 — Outcome 2: Moderate-to-Severe HE vs Mild HE
#           GRADE_HE ∈ {1, ≥2}  |  label = HE_ModerateSevere
# =============================================================================
mask_o2 = df[COL_GRADE].isin([1, 2]) | (df[COL_GRADE] >= 2)
mask_o2 = df[COL_GRADE] >= 1          # patients with GRADE_HE 1 or ≥2
df_o2   = df[mask_o2].reset_index(drop=True)
y_o2    = df_o2[COL_HE_MODSEV].astype(int).values

n_o2       = len(df_o2)
n_o2_pos   = int(y_o2.sum())          # mod-severe HE (positive class)
n_o2_neg   = n_o2 - n_o2_pos          # mild HE       (negative class)
prev_o2    = n_o2_pos / n_o2

print("\n  Outcome 2 — Moderate-to-Severe HE vs Mild HE")
print(f"    Total patients : {n_o2}")
print(f"    Mod-Sev  (y=1) : {n_o2_pos} ({100*prev_o2:.1f}%)")
print(f"    Mild HE  (y=0) : {n_o2_neg} ({100*(1-prev_o2):.1f}%)")

# =============================================================================
# STEP 3 — Feature matrix extraction
# Radiomic feature columns are frozen here from the clean dataset.
# MELD is kept separate and concatenated inside the pipeline where needed.
# =============================================================================
X_o1_rad   = df_o1[rad_cols].values.astype(float)
X_o1_meld  = df_o1[[COL_MELD]].values.astype(float)
X_o1_combo = np.hstack([X_o1_meld, X_o1_rad])

X_o2_rad   = df_o2[rad_cols].values.astype(float)
X_o2_meld  = df_o2[[COL_MELD]].values.astype(float)
X_o2_combo = np.hstack([X_o2_meld, X_o2_rad])

print(f"\n  Feature matrices:")
print(f"    O1 — Radiomic  : {X_o1_rad.shape}")
print(f"    O1 — MELD      : {X_o1_meld.shape}")
print(f"    O1 — Combined  : {X_o1_combo.shape}")
print(f"    O2 — Radiomic  : {X_o2_rad.shape}")
print(f"    O2 — MELD      : {X_o2_meld.shape}")
print(f"    O2 — Combined  : {X_o2_combo.shape}")

# =============================================================================
# STEP 4 — Cross-system scanner labels per subset
# Used in Cell 10 (cross-system validation).
# =============================================================================
scanner_o1 = df_o1[COL_MR_SYSTEM].values  # "1.5T" or "3.0T"
scanner_o2 = df_o2[COL_MR_SYSTEM].values

print(f"\n  Scanner distribution — Outcome 1:")
for s, c in zip(*np.unique(scanner_o1, return_counts=True)):
    print(f"    {s} : {c}")

print(f"\n  Scanner distribution — Outcome 2:")
for s, c in zip(*np.unique(scanner_o2, return_counts=True)):
    print(f"    {s} : {c}")

# =============================================================================
# SUMMARY
# =============================================================================
print(f"\n✅  Cell 3 — Subset creation complete.")
print(f"    Outcome 1 : {n_o1} patients  |  {len(rad_cols)} radiomic features")
print(f"    Outcome 2 : {n_o2} patients  |  {len(rad_cols)} radiomic features")

In [ ]:
# =============================================================================
# Cell 4 — Nested cross-validation: out-of-fold (OOF) predictions
# =============================================================================
# Generates OOF predicted probabilities for all 6 model × outcome
# combinations using nested stratified 5-fold cross-validation.
#
# Design (TRIPOD-AI item 10a):
#   Outer CV : StratifiedKFold(n_splits=5, shuffle=True,
#              random_state=RANDOM_STATE)
#              → produces one OOF predicted probability per subject.
#              → used for all downstream metrics (Cell 5), calibration
#                (Cell 6), DCA (Cell 7), SHAP (Cell 8), nomograms (Cell 11).
#
#   Inner CV : StratifiedKFold(n_splits=5, shuffle=True,
#              random_state=RANDOM_STATE)
#              → GridSearchCV over C ∈ [0.001, 1.0] (10 log-spaced values)
#              → selects optimal L1 regularisation per outer fold.
#
# Pipeline per fold (leakage-safe):
#   1. StandardScaler  — fit on training fold only, applied to val fold.
#   2. LogisticRegression(penalty="l1", solver="liblinear",
#                         class_weight="balanced")
#      — L1 penalty performs implicit feature selection (LASSO).
#      — class_weight="balanced" corrects for class imbalance.
#   3. GridSearchCV   — inner loop selects best C on training fold.
#
# Models:
#   "MELD"      — single predictor (MELD score)
#   "Radiomics" — all IBSI-compliant radiomic features (prefix "original_")
#   "Combined"  — MELD + radiomic features concatenated
#
# Outputs (saved to disk for reproducibility — TRIPOD-AI item 15):
#   outputs/tables/oof_predictions_outcome1.csv
#   outputs/tables/oof_predictions_outcome2.csv
#
# TRIPOD-AI items covered:
#   - 9  : model specification (L1 logistic regression, liblinear)
#   - 10a: validation method (nested stratified 5-fold CV)
#   - 10b: hyperparameter selection (inner GridSearchCV)
#   - 15 : OOF predictions saved for independent verification
# =============================================================================

def make_pipeline():
    """
    Returns a fresh sklearn Pipeline:
      StandardScaler → L1 LogisticRegression (liblinear, balanced).
    Instantiated fresh per outer fold to prevent state leakage.
    """
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    LogisticRegression(
            penalty      = "l1",
            solver       = "liblinear",
            class_weight = "balanced",
            max_iter     = 2000,
            random_state = RANDOM_STATE,
        )),
    ])


def nested_cv_oof(X, y, label):
    """
    Nested stratified 5-fold CV returning OOF predicted probabilities.

    Parameters
    ----------
    X     : np.ndarray, feature matrix (n_samples, n_features)
    y     : np.ndarray, binary outcome  (n_samples,)
    label : str, descriptive label for progress output

    Returns
    -------
    oof_probs : np.ndarray, OOF predicted P(y=1), shape (n_samples,)
    best_Cs   : list of float, best C selected per outer fold
    """
    outer_cv  = StratifiedKFold(
        n_splits     = CV_FOLDS,
        shuffle      = True,
        random_state = RANDOM_STATE,
    )
    inner_cv  = StratifiedKFold(
        n_splits     = CV_FOLDS,
        shuffle      = True,
        random_state = RANDOM_STATE,
    )

    oof_probs = np.zeros(len(y))
    best_Cs   = []

    for fold, (train_idx, val_idx) in enumerate(outer_cv.split(X, y), 1):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        # Inner CV: hyperparameter tuning on training fold only
        pipe   = make_pipeline()
        search = GridSearchCV(
            estimator  = pipe,
            param_grid = C_GRID,
            cv         = inner_cv,
            scoring    = "roc_auc",
            n_jobs     = 1,
        )
        search.fit(X_tr, y_tr)

        best_C = search.best_params_["clf__C"]
        best_Cs.append(best_C)

        # Predict on validation fold using best pipeline
        oof_probs[val_idx] = search.best_estimator_.predict_proba(X_val)[:, 1]

        print(f"    Fold {fold}/5 — best C: {best_C:.4f}  "
              f"| val AUC: "
              f"{roc_auc_score(y_val, oof_probs[val_idx]):.3f}")

    return oof_probs, best_Cs


# =============================================================================
# RUN — Outcome 1
# =============================================================================
print("=" * 60)
print("OUTCOME 1 — Mild HE vs No HE")
print("=" * 60)

print("\n  ▶ MELD")
p_o1_meld,  Cs_o1_meld  = nested_cv_oof(X_o1_meld,  y_o1, "O1-MELD")

print("\n  ▶ Radiomics")
p_o1_rad,   Cs_o1_rad   = nested_cv_oof(X_o1_rad,   y_o1, "O1-Radiomics")

print("\n  ▶ Combined")
p_o1_combo, Cs_o1_combo = nested_cv_oof(X_o1_combo, y_o1, "O1-Combined")

# =============================================================================
# RUN — Outcome 2
# =============================================================================
print("\n" + "=" * 60)
print("OUTCOME 2 — Moderate-to-Severe HE vs Mild HE")
print("=" * 60)

print("\n  ▶ MELD")
p_o2_meld,  Cs_o2_meld  = nested_cv_oof(X_o2_meld,  y_o2, "O2-MELD")

print("\n  ▶ Radiomics")
p_o2_rad,   Cs_o2_rad   = nested_cv_oof(X_o2_rad,   y_o2, "O2-Radiomics")

print("\n  ▶ Combined")
p_o2_combo, Cs_o2_combo = nested_cv_oof(X_o2_combo, y_o2, "O2-Combined")

# =============================================================================
# SAVE OOF PREDICTIONS (TRIPOD-AI item 15)
# =============================================================================
oof_o1 = pd.DataFrame({
    "y_true":     y_o1,
    "p_meld":     p_o1_meld,
    "p_rad":      p_o1_rad,
    "p_combo":    p_o1_combo,
    COL_MR_SYSTEM: scanner_o1,
})
oof_o1.to_csv(TAB_DIR / "oof_predictions_outcome1.csv", index=False)

oof_o2 = pd.DataFrame({
    "y_true":     y_o2,
    "p_meld":     p_o2_meld,
    "p_rad":      p_o2_rad,
    "p_combo":    p_o2_combo,
    COL_MR_SYSTEM: scanner_o2,
})
oof_o2.to_csv(TAB_DIR / "oof_predictions_outcome2.csv", index=False)

print(f"\n  ✓ OOF predictions saved:")
print(f"    → {(TAB_DIR / 'oof_predictions_outcome1.csv').resolve()}")
print(f"    → {(TAB_DIR / 'oof_predictions_outcome2.csv').resolve()}")

# =============================================================================
# OVERALL OOF AUC SUMMARY
# =============================================================================
print("\n" + "=" * 60)
print("OOF AUC SUMMARY")
print("=" * 60)
for outcome, y, pm, pr, pc in [
    ("Outcome 1", y_o1, p_o1_meld, p_o1_rad, p_o1_combo),
    ("Outcome 2", y_o2, p_o2_meld, p_o2_rad, p_o2_combo),
]:
    print(f"\n  {outcome}")
    for label, p in zip(MODEL_LABELS, [pm, pr, pc]):
        print(f"    {label:<12} AUC = {roc_auc_score(y, p):.3f}")

print(f"\n✅  Cell 4 — Nested CV and OOF predictions complete.")

In [ ]:
# =============================================================================
# Cell 5 — Performance metrics, bootstrap CI, DeLong test
#           → Table 2 (Outcome 1) and Table 3 (Outcome 2)
# =============================================================================
# Computes for each model × outcome combination:
#
#   - AUC              : point estimate + 95% percentile bootstrap CI
#                        (n = 2,000 resamples, seed = BOOT_SEED)
#   - Brier score      : mean squared prediction error
#   - Calibration slope: slope of logistic regression of y on logit(p̂)
#                        95% CI via bootstrap
#   - Cal. intercept   : intercept of same model; 95% CI via bootstrap
#                        perfect calibration: slope = 1, intercept = 0
#   - DeLong test      : two-tailed paired comparison of each model vs
#                        MELD (reference); implemented via the
#                        Mann-Whitney U statistic on placement values
#                        (DeLong et al., Biometrics 1988)
#
# All metrics are computed on OOF predictions from Cell 4 to avoid
# optimistic bias (TRIPOD-AI item 10a).
#
# Outputs:
#   outputs/tables/performance_metrics.csv  — machine-readable
#   Inline HTML table                       — publication-ready
#
# TRIPOD-AI items covered:
#   - 10a: performance on validation set (OOF)
#   - 11 : uncertainty quantification (bootstrap CI)
#   - 12 : statistical comparison (DeLong test)
# =============================================================================

# =============================================================================
# HELPER — DeLong test
# Two-tailed AUC comparison via placement values (DeLong et al. 1988).
# =============================================================================

def auc_placement_values(y, p):
    """
    Compute placement values (V10, V01) for DeLong variance estimation.

    Returns
    -------
    V10 : np.ndarray, placement values for positive class
    V01 : np.ndarray, placement values for negative class
    """
    pos = p[y == 1]
    neg = p[y == 0]
    n1, n0 = len(pos), len(neg)

    V10 = np.array([
        (np.sum(pj > neg) + 0.5 * np.sum(pj == neg)) / n0
        for pj in pos
    ])
    V01 = np.array([
        (np.sum(pi < pos) + 0.5 * np.sum(pi == pos)) / n1
        for pi in neg
    ])
    return V10, V01


def delong_test(y, p1, p2):
    """
    Two-tailed DeLong test comparing AUC(p1) vs AUC(p2).

    Parameters
    ----------
    y  : np.ndarray, binary outcome
    p1 : np.ndarray, predicted probabilities model 1
    p2 : np.ndarray, predicted probabilities model 2

    Returns
    -------
    z_stat : float
    p_value: float (two-tailed)
    """
    y  = np.asarray(y)
    p1 = np.asarray(p1)
    p2 = np.asarray(p2)

    n1 = int(y.sum())
    n0 = int((1 - y).sum())

    V10_1, V01_1 = auc_placement_values(y, p1)
    V10_2, V01_2 = auc_placement_values(y, p2)

    auc1 = V10_1.mean()
    auc2 = V10_2.mean()

    # Covariance matrix of (AUC1, AUC2)
    S10 = np.cov(np.vstack([V10_1, V10_2]))   # 2×2
    S01 = np.cov(np.vstack([V01_1, V01_2]))   # 2×2

    var_diff = (S10[0, 0] / n1 + S01[0, 0] / n0
                + S10[1, 1] / n1 + S01[1, 1] / n0
                - 2 * (S10[0, 1] / n1 + S01[0, 1] / n0))

    if var_diff <= 0:
        return 0.0, 1.0

    z     = (auc1 - auc2) / np.sqrt(var_diff)
    p_val = 2 * (1 - stats.norm.cdf(abs(z)))
    return float(z), float(p_val)


# =============================================================================
# HELPER — Bootstrap CI for AUC, Brier, calibration slope/intercept
# =============================================================================

def bootstrap_metrics(y, p, n_boot=N_BOOT, seed=BOOT_SEED):
    """
    Percentile bootstrap CI (95%) for AUC, Brier score,
    calibration slope, and calibration intercept.

    Parameters
    ----------
    y      : np.ndarray, binary outcome
    p      : np.ndarray, predicted probabilities
    n_boot : int, number of bootstrap resamples
    seed   : int, random seed

    Returns
    -------
    dict with point estimates and 95% CI bounds.
    """
    rng = np.random.default_rng(seed)

    auc_b, brier_b, slope_b, intercept_b = [], [], [], []

    eps = 1e-6
    p_c = np.clip(p, eps, 1 - eps)

    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        yb, pb = y[idx], p_c[idx]

        # Skip degenerate resamples (only one class)
        if len(np.unique(yb)) < 2:
            continue

        auc_b.append(roc_auc_score(yb, pb))
        brier_b.append(brier_score_loss(yb, pb))

        # Calibration: logistic regression of y on logit(p̂)
        logit_pb = np.log(pb / (1 - pb))
        try:
            lr = LogisticRegression(penalty=None, solver="lbfgs",
                                    max_iter=500)
            lr.fit(logit_pb.reshape(-1, 1), yb)
            slope_b.append(float(lr.coef_[0][0]))
            intercept_b.append(float(lr.intercept_[0]))
        except Exception:
            pass

    def ci(arr):
        arr = np.array(arr)
        return float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5))

    # Point estimates on full OOF data
    logit_p  = np.log(p_c / (1 - p_c))
    lr_cal   = LogisticRegression(penalty=None, solver="lbfgs", max_iter=500)
    lr_cal.fit(logit_p.reshape(-1, 1), y)
    cal_slope     = float(lr_cal.coef_[0][0])
    cal_intercept = float(lr_cal.intercept_[0])

    auc_lo,  auc_hi  = ci(auc_b)
    brier_lo, brier_hi = ci(brier_b)
    slope_lo, slope_hi = ci(slope_b)
    int_lo,   int_hi   = ci(intercept_b)

    return {
        "AUC":           roc_auc_score(y, p),
        "AUC_CI_lo":     auc_lo,
        "AUC_CI_hi":     auc_hi,
        "Brier":         brier_score_loss(y, p_c),
        "Brier_CI_lo":   brier_lo,
        "Brier_CI_hi":   brier_hi,
        "Cal_Slope":     cal_slope,
        "Slope_CI_lo":   slope_lo,
        "Slope_CI_hi":   slope_hi,
        "Cal_Intercept": cal_intercept,
        "Int_CI_lo":     int_lo,
        "Int_CI_hi":     int_hi,
    }


# =============================================================================
# COMPUTE METRICS — all 6 model × outcome combinations
# =============================================================================
TASKS = [
    ("Outcome 1 — Mild vs No HE",         y_o1,
     p_o1_meld, p_o1_rad, p_o1_combo),
    ("Outcome 2 — Mod-Severe vs Mild HE",  y_o2,
     p_o2_meld, p_o2_rad, p_o2_combo),
]

rows = []

for outcome_label, y, p_meld, p_rad, p_combo in TASKS:
    print(f"\n  Computing metrics — {outcome_label}")

    for model_label, p_model in zip(
        MODEL_LABELS, [p_meld, p_rad, p_combo]
    ):
        m = bootstrap_metrics(y, p_model)

        # DeLong test vs MELD reference
        if model_label == "MELD":
            z_stat, p_delong = np.nan, np.nan
            sig = "—"
        else:
            z_stat, p_delong = delong_test(y, p_model, p_meld)
            if   p_delong < 0.001: sig = "***"
            elif p_delong < 0.01:  sig = "**"
            elif p_delong < 0.05:  sig = "*"
            else:                  sig = "ns"

        rows.append({
            "Outcome":       outcome_label,
            "Model":         model_label,
            "AUC":           m["AUC"],
            "AUC_CI_lo":     m["AUC_CI_lo"],
            "AUC_CI_hi":     m["AUC_CI_hi"],
            "Brier":         m["Brier"],
            "Brier_CI_lo":   m["Brier_CI_lo"],
            "Brier_CI_hi":   m["Brier_CI_hi"],
            "Cal_Slope":     m["Cal_Slope"],
            "Slope_CI_lo":   m["Slope_CI_lo"],
            "Slope_CI_hi":   m["Slope_CI_hi"],
            "Cal_Intercept": m["Cal_Intercept"],
            "Int_CI_lo":     m["Int_CI_lo"],
            "Int_CI_hi":     m["Int_CI_hi"],
            "DeLong_z":      z_stat,
            "DeLong_p":      p_delong,
            "Sig":           sig,
        })
        print(f"    {model_label:<12} AUC {m['AUC']:.3f} "
              f"({m['AUC_CI_lo']:.3f}–{m['AUC_CI_hi']:.3f})  "
              f"Brier {m['Brier']:.3f}  "
              f"Slope {m['Cal_Slope']:.3f}  "
              f"DeLong p: "
              f"{'—' if np.isnan(p_delong) else f'{p_delong:.3f}'} {sig}")

# =============================================================================
# SAVE — machine-readable CSV
# =============================================================================
df_metrics = pd.DataFrame(rows)
df_metrics.to_csv(TAB_DIR / "performance_metrics.csv", index=False)
print(f"\n  ✓ Metrics saved → {(TAB_DIR / 'performance_metrics.csv').resolve()}")

# =============================================================================
# DISPLAY — inline HTML publication table
# =============================================================================
def format_ci(lo, hi):
    return f"({lo:.3f}–{hi:.3f})"

def p_fmt(p):
    if np.isnan(p): return "—"
    if p < 0.001:   return "< 0.001"
    if p < 0.01:    return "< 0.01"
    return f"{p:.3f}"

th = ("padding:6px 14px; text-align:center; font-weight:normal; "
      "border-top:2px solid #111; border-bottom:1px solid #111; "
      "background:#f7f7f7;")
th_l = th.replace("center", "left")
td   = "padding:5px 14px; text-align:center;"
td_l = td.replace("center", "left")

html = """
<div style="font-family:'Helvetica Neue',Arial,sans-serif;
            font-size:12px; max-width:980px; margin-top:12px;">"""

for outcome_label in df_metrics["Outcome"].unique():
    df_out = df_metrics[df_metrics["Outcome"] == outcome_label]
    tnum   = "2" if "Outcome 1" in outcome_label else "3"

    html += f"""
  <p style="margin:16px 0 4px 0; font-size:12px;">
    <b>Table {tnum} — {outcome_label}</b>
  </p>
  <table style="border-collapse:collapse; width:100%; margin-bottom:20px;">
    <thead>
      <tr>
        <th style="{th_l}">Model</th>
        <th style="{th}">AUC (95% CI)</th>
        <th style="{th}">Brier (95% CI)</th>
        <th style="{th}">Cal. Slope (95% CI)</th>
        <th style="{th}">Cal. Intercept (95% CI)</th>
        <th style="{th}">DeLong <i>p</i></th>
        <th style="{th}">Sig.</th>
      </tr>
    </thead>
    <tbody>"""

    for _, r in df_out.iterrows():
        bold_o = "<b>" if r.Model == "Combined" else ""
        bold_c = "</b>" if r.Model == "Combined" else ""
        color  = f"color:{MODEL_COLORS[r.Model]};"

        html += f"""
      <tr style="border-bottom:0.5px solid #e8e8e8;">
        <td style="{td_l}{color}">{bold_o}{r.Model}{bold_c}</td>
        <td style="{td}">{bold_o}{r.AUC:.3f} {format_ci(r.AUC_CI_lo, r.AUC_CI_hi)}{bold_c}</td>
        <td style="{td}">{r.Brier:.3f} {format_ci(r.Brier_CI_lo, r.Brier_CI_hi)}</td>
        <td style="{td}">{r.Cal_Slope:.3f} {format_ci(r.Slope_CI_lo, r.Slope_CI_hi)}</td>
        <td style="{td}">{r.Cal_Intercept:.3f} {format_ci(r.Int_CI_lo, r.Int_CI_hi)}</td>
        <td style="{td}">{p_fmt(r.DeLong_p)}</td>
        <td style="{td}">{r.Sig}</td>
      </tr>"""

    html += f"""
    </tbody>
    <tfoot>
      <tr style="border-top:2px solid #111;">
        <td colspan="7" style="padding:4px 14px; font-size:11px; color:#555;">
          AUC = area under the ROC curve; CI = 95% bootstrap confidence
          interval (n = {N_BOOT} resamples); Brier = mean squared prediction
          error; Cal. Slope and Intercept from logistic regression of outcome
          on logit(predicted probability); DeLong <i>p</i> = two-tailed
          comparison vs MELD. * p&lt;0.05, ** p&lt;0.01, *** p&lt;0.001,
          ns = not significant.
        </td>
      </tr>
    </tfoot>
  </table>"""

html += "\n</div>"
display(HTML(html))

print(f"\n✅  Cell 5 — Performance metrics complete.")

In [ ]:
# =============================================================================
# Cell 6 — Calibration plots (reliability diagrams)
#           Outcome 1 and Outcome 2
# =============================================================================
# Produces reliability diagrams for all 6 model × outcome combinations.
#
# Method:
#   - Predictions are binned into equal-frequency quantile bins (n = 10)
#     to ensure each bin contains a comparable number of observations,
#     avoiding empty bins at extreme probabilities.
#   - Each bin shows: observed event rate (y-axis) vs mean predicted
#     probability (x-axis).
#   - The diagonal reference line represents perfect calibration
#     (predicted = observed).
#   - Calibration slope and intercept (from Cell 5) are annotated
#     on each panel.
#   - A marginal rug plot along the x-axis shows the distribution
#     of individual predicted probabilities.
#
# Layout:
#   One figure per outcome, 3 panels side by side (one per model).
#   Shared axes for direct visual comparison across models.
#
# Outputs:
#   outputs/figures/calibration/calibration_outcome1.png / .pdf / .svg
#   outputs/figures/calibration/calibration_outcome2.png / .pdf / .svg
#
# TRIPOD-AI items covered:
#   - 10a: calibration assessment on OOF predictions
#   - 11 : visual calibration reporting
# =============================================================================

N_BINS = 10   # equal-frequency quantile bins

def calibration_curve_quantile(y, p, n_bins=N_BINS):
    """
    Compute calibration curve using equal-frequency (quantile) binning.

    Parameters
    ----------
    y      : np.ndarray, binary outcome
    p      : np.ndarray, predicted probabilities
    n_bins : int, number of bins

    Returns
    -------
    mean_pred : np.ndarray, mean predicted probability per bin
    frac_pos  : np.ndarray, observed event rate per bin
    bin_sizes : np.ndarray, number of observations per bin
    """
    quantiles  = np.percentile(p, np.linspace(0, 100, n_bins + 1))
    quantiles  = np.unique(quantiles)   # remove duplicates at boundaries

    mean_pred, frac_pos, bin_sizes = [], [], []

    for i in range(len(quantiles) - 1):
        lo, hi = quantiles[i], quantiles[i + 1]
        # Include right boundary in last bin
        if i == len(quantiles) - 2:
            mask = (p >= lo) & (p <= hi)
        else:
            mask = (p >= lo) & (p < hi)
        if mask.sum() == 0:
            continue
        mean_pred.append(p[mask].mean())
        frac_pos.append(y[mask].mean())
        bin_sizes.append(mask.sum())

    return (np.array(mean_pred),
            np.array(frac_pos),
            np.array(bin_sizes))


def plot_calibration_outcome(outcome_label, y,
                             p_meld, p_rad, p_combo,
                             df_metrics, fname):
    """
    Three-panel calibration figure for one outcome.

    Parameters
    ----------
    outcome_label : str
    y             : np.ndarray, binary outcome
    p_meld/rad/combo : np.ndarray, OOF predicted probabilities
    df_metrics    : pd.DataFrame, from Cell 5 (for slope/intercept annotation)
    fname         : str, output filename stem
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
    fig.suptitle(f"Calibration — {outcome_label}",
                 fontsize=13, fontweight="normal", y=1.01)

    models = [
        ("MELD",      p_meld,  MODEL_COLORS["MELD"]),
        ("Radiomics", p_rad,   MODEL_COLORS["Radiomics"]),
        ("Combined",  p_combo, MODEL_COLORS["Combined"]),
    ]

    for ax, (model_label, p, color) in zip(axes, models):

        mean_pred, frac_pos, bin_sizes = calibration_curve_quantile(y, p)

        # Retrieve calibration metrics from Cell 5
        row = df_metrics[
            (df_metrics["Outcome"].str.contains(
                "Outcome 1" if "Outcome 1" in outcome_label else "Outcome 2"))
            & (df_metrics["Model"] == model_label)
        ]
        slope     = float(row["Cal_Slope"].values[0])
        intercept = float(row["Cal_Intercept"].values[0])
        brier     = float(row["Brier"].values[0])

        # --- Diagonal reference ---
        ax.plot([0, 1], [0, 1],
                color="#999999", linewidth=1.0,
                linestyle="--", label="Perfect calibration")

        # --- Calibration curve ---
        ax.plot(mean_pred, frac_pos,
                color=color, linewidth=2.0,
                marker="o", markersize=5,
                label=model_label)

        # --- Confidence band (Wilson interval per bin) ---
        for mp, fp, n in zip(mean_pred, frac_pos, bin_sizes):
            z    = 1.96
            denom = 1 + z**2 / n
            centre = (fp + z**2 / (2 * n)) / denom
            half   = z * np.sqrt(fp * (1 - fp) / n + z**2 / (4 * n**2)) / denom
            ax.plot([mp, mp],
                    [max(0, centre - half), min(1, centre + half)],
                    color=color, linewidth=1.0, alpha=0.5)

        # --- Rug plot (predicted probability distribution) ---
        ax.plot(p, np.full_like(p, -0.03),
                "|", color=color, alpha=0.3,
                markersize=4, markeredgewidth=0.6)

        # --- Annotation: slope, intercept, Brier ---
        ax.text(0.04, 0.93,
                f"Slope:      {slope:.3f}\n"
                f"Intercept: {intercept:.3f}\n"
                f"Brier:       {brier:.3f}",
                transform=ax.transAxes,
                fontsize=8.5, va="top",
                family="monospace",
                bbox=dict(boxstyle="round,pad=0.3",
                          facecolor="white",
                          edgecolor="#cccccc",
                          alpha=0.85))

        # --- Formatting ---
        ax.set_xlim(-0.02, 1.02)
        ax.set_ylim(-0.08, 1.08)
        ax.set_xlabel("Mean predicted probability", fontsize=10)
        if ax == axes[0]:
            ax.set_ylabel("Observed event rate", fontsize=10)
        ax.set_title(model_label, fontsize=11,
                     fontweight="normal",
                     color=color)
        ax.grid(True, linestyle="--", alpha=0.25)
        ax.axhline(0, color="#cccccc", linewidth=0.6)
        ax.axvline(0, color="#cccccc", linewidth=0.6)

    plt.tight_layout()

    for ext in ("png", "pdf", "svg"):
        fig.savefig(CAL_DIR / f"{fname}.{ext}",
                    dpi=300, bbox_inches="tight",
                    facecolor="white")

    plt.show()
    plt.close(fig)
    print(f"  ✓ Saved → {(CAL_DIR / fname)}.png / .pdf / .svg")


# =============================================================================
# RENDER — Outcome 1
# =============================================================================
print("▶  Calibration — Outcome 1")
plot_calibration_outcome(
    outcome_label = "Outcome 1 — Mild vs No HE",
    y             = y_o1,
    p_meld        = p_o1_meld,
    p_rad         = p_o1_rad,
    p_combo       = p_o1_combo,
    df_metrics    = df_metrics,
    fname         = "calibration_outcome1",
)

# =============================================================================
# RENDER — Outcome 2
# =============================================================================
print("\n▶  Calibration — Outcome 2")
plot_calibration_outcome(
    outcome_label = "Outcome 2 — Moderate-Severe vs Mild HE",
    y             = y_o2,
    p_meld        = p_o2_meld,
    p_rad         = p_o2_rad,
    p_combo       = p_o2_combo,
    df_metrics    = df_metrics,
    fname         = "calibration_outcome2",
)

print(f"\n✅  Cell 6 — Calibration plots complete.")

In [ ]:
# =============================================================================
# Cell 7 — Decision Curve Analysis (DCA)
#           Outcome 1 and Outcome 2
# =============================================================================
# Decision Curve Analysis quantifies the net clinical benefit of each
# model across a range of threshold probabilities, compared with two
# reference strategies:
#   - "Treat none" : net benefit = 0 by definition
#   - "Treat all"  : all patients assumed positive
#
# Net benefit formula (Vickers & Elkin, Med Decis Making 2006):
#   NB(t) = (TP/n) - (FP/n) × (t / (1 − t))
#   where t = threshold probability, n = total patients
#
# Curves are plotted as step functions (no smoothing) to reflect the
# true net benefit at each discrete threshold — TRIPOD-AI item 10b.
# A model provides clinical utility where its curve exceeds both
# reference strategies across the clinically relevant threshold range
# (5%–50% per the manuscript).
#
# An inline HTML table reports net benefit at three clinically
# relevant thresholds (10%, 20%, 30%) for each model.
#
# Colours:
#   MELD      → green  #079A02
#   Radiomics → blue   #005DCE
#   Combined  → red    #A60628
#
# Outputs:
#   outputs/figures/dca/dca_outcome1.png / .pdf / .svg
#   outputs/figures/dca/dca_outcome2.png / .pdf / .svg
#
# TRIPOD-AI items covered:
#   - 10b: clinical utility assessment
#   - 12 : decision analytic methods (Vickers & Elkin 2006)
# =============================================================================

# =============================================================================
# NET BENEFIT FUNCTIONS
# =============================================================================

def net_benefit(y, p, thresholds):
    """
    Compute net benefit at each threshold probability.
    Step function — no smoothing or interpolation.

    Parameters
    ----------
    y          : np.ndarray, binary outcome
    p          : np.ndarray, predicted probabilities
    thresholds : np.ndarray, threshold values in (0, 1)

    Returns
    -------
    nb : np.ndarray, net benefit per threshold
    """
    y, p = np.asarray(y).flatten(), np.asarray(p).flatten()
    n    = len(y)
    nb   = np.zeros(len(thresholds))
    for i, t in enumerate(thresholds):
        pos = p >= t
        tp  = np.sum(pos & (y == 1))
        fp  = np.sum(pos & (y == 0))
        nb[i] = (tp / n) - (fp / n) * (t / (1 - t))
    return nb


def net_benefit_treat_all(y, thresholds):
    """
    Net benefit of the treat-all strategy at each threshold.
    Equivalent to assuming every patient is positive.
    """
    y  = np.asarray(y).flatten()
    n  = len(y)
    nb = np.zeros(len(thresholds))
    for i, t in enumerate(thresholds):
        tp    = np.sum(y == 1)
        fp    = np.sum(y == 0)
        nb[i] = (tp / n) - (fp / n) * (t / (1 - t))
    return nb


# =============================================================================
# PLOT FUNCTION
# =============================================================================

def plot_dca(outcome_label, y,
             p_meld, p_rad, p_combo,
             fname,
             threshold_range=(0.01, 0.95),
             highlight_range=(0.05, 0.50)):
    """
    Decision curve plot with step lines, dotted grid, shadowed legend.
    Clinically relevant range is highlighted with a shaded band.

    Parameters
    ----------
    threshold_range  : tuple, full x-axis range for computation
    highlight_range  : tuple, clinically relevant range (shaded)
                       per manuscript: 5%–50%
    """
    thresholds = np.linspace(
        threshold_range[0], threshold_range[1], 200
    )

    nb_meld  = net_benefit(y, p_meld,  thresholds)
    nb_rad   = net_benefit(y, p_rad,   thresholds)
    nb_combo = net_benefit(y, p_combo, thresholds)
    nb_all   = net_benefit_treat_all(y, thresholds)
    nb_none  = np.zeros(len(thresholds))

    fig, ax = plt.subplots(figsize=(9, 7))

    # --- Clinically relevant range shading ---
    ax.axvspan(highlight_range[0], highlight_range[1],
               alpha=0.06, color="#444444",
               label=f"Clinical range ({int(highlight_range[0]*100)}%"
                     f"–{int(highlight_range[1]*100)}%)")

    # --- Reference strategies ---
    ax.axhline(y=0, color="gray", lw=1.0,
               linestyle="-", label="Treat none")
    ax.step(thresholds, nb_all,
            color="black", lw=1.2,
            linestyle="--", where="mid",
            label="Treat all")

    # --- Models (step lines) ---
    ax.step(thresholds, nb_meld,
            color=MODEL_COLORS["MELD"],
            lw=2.5, where="mid", label="MELD")
    ax.step(thresholds, nb_rad,
            color=MODEL_COLORS["Radiomics"],
            lw=2.5, where="mid", label="Radiomics")
    ax.step(thresholds, nb_combo,
            color=MODEL_COLORS["Combined"],
            lw=2.5, where="mid", label="Combined")

    # --- Axes limits ---
    ax.set_xlim(threshold_range[0], threshold_range[1])
    y_min = min(nb_meld.min(), nb_rad.min(),
                nb_combo.min(), nb_all.min())
    ax.set_ylim(max(-0.05, y_min - 0.02),
                max(nb_combo.max(), nb_all.max()) + 0.15)

    # --- Labels & formatting ---
    ax.set_xlabel("Threshold Probability", fontsize=12)
    ax.set_ylabel("Net Benefit", fontsize=12)
    ax.set_title(f"Decision Curve Analysis — {outcome_label}",
                 fontweight="bold", fontsize=13)
    ax.legend(loc="upper right", frameon=True,
              shadow=True, fontsize=10)
    ax.grid(alpha=0.3, linestyle=":")

    plt.tight_layout()

    for ext in ("png", "pdf", "svg"):
        fig.savefig(DCA_DIR / f"{fname}.{ext}",
                    dpi=300, bbox_inches="tight",
                    facecolor="white")

    plt.show()
    plt.close(fig)
    print(f"  ✓ Saved → {(DCA_DIR / fname)}.png / .pdf / .svg")

    return thresholds, nb_meld, nb_rad, nb_combo, nb_all


# =============================================================================
# INLINE LEGEND FUNCTION
# =============================================================================

def print_dca_legend(outcome_label, fname,
                     thresholds, nb_meld, nb_rad,
                     nb_combo, nb_all,
                     ref_thresholds=(0.10, 0.20, 0.30)):
    """
    Inline HTML legend (Hepatology style) with net benefit values
    at clinically relevant threshold probabilities.
    """
    nb_none = np.zeros(len(thresholds))

    th  = ("padding:5px 12px; text-align:center; font-weight:normal; "
           "border-top:2px solid #111; border-bottom:1px solid #111; "
           "background:#f7f7f7;")
    th_l = th.replace("center", "left")
    td   = "padding:5px 12px; text-align:center;"
    td_l = td.replace("center", "left")

    html = f"""
    <div style="font-family:'Helvetica Neue',Arial,sans-serif;
                font-size:12px; max-width:860px;
                margin-bottom:28px; margin-top:8px;">

      <p style="font-size:12px; margin:0 0 6px 0;">
        <b>Figure legend — Decision Curve Analysis ({outcome_label})</b>
      </p>

      <p style="margin:0 0 10px 0; color:#333; line-height:1.6;">
        Decision curves showing the net benefit of the MELD, Radiomics,
        and Combined models across a range of threshold probabilities,
        compared with the reference strategies of treating all patients
        and treating none. Net benefit is computed as the true-positive
        rate minus the false-positive rate weighted by the odds of the
        threshold probability (Vickers &amp; Elkin,
        <i>Med Decis Making</i> 2006;26:565–574).
        Curves are plotted as step functions without smoothing.
        The shaded band indicates the clinically relevant threshold
        range (5%–50%) as reported in the manuscript.
        A model is clinically useful where its net benefit exceeds
        both reference strategies. Colors: MELD = green,
        Radiomics = blue, Combined = red.
      </p>

      <table style="border-collapse:collapse; width:100%;
                    margin-bottom:10px;">
        <thead>
          <tr>
            <th style="{th_l}">Strategy / Model</th>
            {"".join(
                f'<th style="{th}">t = {int(t*100)}%</th>'
                for t in ref_thresholds
            )}
          </tr>
        </thead>
        <tbody>"""

    curves = [
        ("Treat none",  nb_none,  None),
        ("Treat all",   nb_all,   None),
        ("MELD",        nb_meld,  MODEL_COLORS["MELD"]),
        ("Radiomics",   nb_rad,   MODEL_COLORS["Radiomics"]),
        ("Combined",    nb_combo, MODEL_COLORS["Combined"]),
    ]

    for label, nb, color in curves:
        bold_o  = "<b>" if label == "Combined" else ""
        bold_c  = "</b>" if label == "Combined" else ""
        c_style = f"color:{color};" if color else ""
        cells   = ""
        for t in ref_thresholds:
            idx    = np.argmin(np.abs(thresholds - t))
            cells += (f'<td style="{td}">'
                      f'{bold_o}{nb[idx]:.4f}{bold_c}</td>')
        html += f"""
          <tr style="border-bottom:0.5px solid #e8e8e8;">
            <td style="{td_l}{c_style}">{bold_o}{label}{bold_c}</td>
            {cells}
          </tr>"""

    html += f"""
        </tbody>
        <tfoot>
          <tr style="border-top:2px solid #111;">
            <td colspan="{1 + len(ref_thresholds)}"
                style="padding:4px 12px; font-size:11px; color:#555;">
              Net benefit at threshold probabilities of
              {", ".join(f"{int(t*100)}%" for t in ref_thresholds)}.
              Saved: <code>{fname}.png / .pdf / .svg</code>
            </td>
          </tr>
        </tfoot>
      </table>
    </div>"""

    display(HTML(html))


# =============================================================================
# RENDER — Outcome 1
# =============================================================================
print("▶  DCA — Outcome 1")
thresholds, nb_meld, nb_rad, nb_combo, nb_all = plot_dca(
    outcome_label = "Outcome 1 — Mild vs No HE",
    y             = y_o1,
    p_meld        = p_o1_meld,
    p_rad         = p_o1_rad,
    p_combo       = p_o1_combo,
    fname         = "dca_outcome1",
)
print_dca_legend(
    outcome_label  = "Outcome 1 — Mild vs No HE",
    fname          = "dca_outcome1",
    thresholds     = thresholds,
    nb_meld        = nb_meld,
    nb_rad         = nb_rad,
    nb_combo       = nb_combo,
    nb_all         = nb_all,
    ref_thresholds = (0.10, 0.20, 0.30),
)

# =============================================================================
# RENDER — Outcome 2
# =============================================================================
print("▶  DCA — Outcome 2")
thresholds, nb_meld, nb_rad, nb_combo, nb_all = plot_dca(
    outcome_label = "Outcome 2 — Moderate-Severe vs Mild HE",
    y             = y_o2,
    p_meld        = p_o2_meld,
    p_rad         = p_o2_rad,
    p_combo       = p_o2_combo,
    fname         = "dca_outcome2",
)
print_dca_legend(
    outcome_label  = "Outcome 2 — Moderate-Severe vs Mild HE",
    fname          = "dca_outcome2",
    thresholds     = thresholds,
    nb_meld        = nb_meld,
    nb_rad         = nb_rad,
    nb_combo       = nb_combo,
    nb_all         = nb_all,
    ref_thresholds = (0.10, 0.20, 0.30),
)

print(f"\n✅  Cell 7 — Decision Curve Analysis complete.")

In [ ]:
# =============================================================================
# Cell 8 — SHAP feature importance
#           Outcome 1 and Outcome 2 (Radiomics model)
# =============================================================================
# Computes SHAP values for the Radiomics-only L1 logistic regression
# model trained on the full dataset (not OOF) for interpretability.
#
# Why full-data model for SHAP:
#   SHAP is used here for feature importance visualisation, not for
#   performance estimation. Training on the full dataset maximises
#   the stability of coefficient estimates and therefore of SHAP
#   attributions. Performance estimates are reported exclusively
#   from OOF predictions (Cell 5) — TRIPOD-AI item 10a.
#
# Method — LinearExplainer:
#   Exact SHAP values for linear models via the closed-form expression:
#     φ_j = w_j × (x_j − E[x_j])
#   where w_j is the scaled coefficient and x_j the scaled feature value.
#   feature_perturbation = "interventional" removes the assumption of
#   feature independence used in the correlation-aware variant.
#
# Hyperparameter:
#   The best C is selected as the median of the best_C values from the
#   outer CV folds (Cell 4), providing a stable regularisation estimate
#   without refitting the inner loop on the full dataset.
#
# Visualisation:
#   - Beeswarm plot (summary_plot): top-20 features by mean |SHAP|.
#     Each dot = one patient; colour = feature value (blue low, red high).
#   - Bar chart: mean |SHAP| per feature (top-20), colour-coded by model.
#   Both plots saved as PNG / PDF.
#
# Outputs:
#   outputs/figures/shap/shap_beeswarm_outcome1.png / .pdf
#   outputs/figures/shap/shap_beeswarm_outcome2.png / .pdf
#   outputs/figures/shap/shap_bar_outcome1.png / .pdf
#   outputs/figures/shap/shap_bar_outcome2.png / .pdf
#   outputs/tables/shap_features_outcome1.csv
#   outputs/tables/shap_features_outcome2.csv
#
# TRIPOD-AI items covered:
#   - 10c: model interpretability
#   - 13a: predictor contribution reporting (SHAP)
# =============================================================================

TOP_N_SHAP = 20   # number of top features to display

def run_shap(X_rad, y, best_Cs, outcome_label, fname_stem):
    """
    Fit full-data radiomics model, compute SHAP values, produce
    beeswarm and bar plots, save CSV of feature importances.

    Parameters
    ----------
    X_rad         : np.ndarray, raw (unscaled) radiomic feature matrix
    y             : np.ndarray, binary outcome
    best_Cs       : list of float, best C per outer fold (from Cell 4)
    outcome_label : str
    fname_stem    : str, filename stem (no extension)
    """

    # -------------------------------------------------------------------------
    # STEP 1 — Select regularisation: median of outer-fold best C values
    # -------------------------------------------------------------------------
    best_C = float(np.median(best_Cs))
    print(f"  Best C (median of outer folds): {best_C:.5f}")

    # -------------------------------------------------------------------------
    # STEP 2 — Fit full-data pipeline (scale + L1 logistic regression)
    # -------------------------------------------------------------------------
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    LogisticRegression(
            penalty      = "l1",
            C            = best_C,
            solver       = "liblinear",
            class_weight = "balanced",
            max_iter     = 2000,
            random_state = RANDOM_STATE,
        )),
    ])
    pipe.fit(X_rad, y)

    # Scaled feature matrix used by LinearExplainer
    X_scaled = pipe.named_steps["scaler"].transform(X_rad)
    model    = pipe.named_steps["clf"]

    # -------------------------------------------------------------------------
    # STEP 3 — Identify non-zero features (selected by LASSO)
    # -------------------------------------------------------------------------
    coefs       = model.coef_[0]
    nonzero_idx = np.where(coefs != 0)[0]
    n_selected  = len(nonzero_idx)
    print(f"  LASSO-selected features: {n_selected} / {len(rad_cols)}")

    # -------------------------------------------------------------------------
    # STEP 4 — SHAP LinearExplainer (interventional perturbation)
    # -------------------------------------------------------------------------
    explainer   = shap.LinearExplainer(
        model,
        X_scaled,
        feature_perturbation = "interventional",
    )
    shap_values = explainer.shap_values(X_scaled)  # shape: (n, p)

    # -------------------------------------------------------------------------
    # STEP 5 — Rank features by mean |SHAP|
    # -------------------------------------------------------------------------
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    ranked_idx    = np.argsort(mean_abs_shap)[::-1]
    top_idx       = ranked_idx[:TOP_N_SHAP]

    feature_names = np.array(rad_cols)

    # Save feature importance table
    shap_df = pd.DataFrame({
        "Feature":       feature_names[ranked_idx],
        "Mean_Abs_SHAP": mean_abs_shap[ranked_idx],
        "LASSO_Coef":    coefs[ranked_idx],
        "Selected":      coefs[ranked_idx] != 0,
    })
    csv_path = TAB_DIR / f"shap_features_{fname_stem}.csv"
    shap_df.to_csv(csv_path, index=False)
    print(f"  ✓ Feature table saved → {csv_path.resolve()}")

    # -------------------------------------------------------------------------
    # STEP 6 — Beeswarm plot (summary_plot)
    # -------------------------------------------------------------------------
    fig_bee, ax_bee = plt.subplots(figsize=(10, 7))
    shap.summary_plot(
        shap_values[:, top_idx],
        X_scaled[:, top_idx],
        feature_names  = feature_names[top_idx].tolist(),
        plot_type      = "dot",
        max_display    = TOP_N_SHAP,
        show           = False,
        color_bar_label= "Feature value",
    )
    plt.title(
        f"SHAP Feature Importance — {outcome_label}\n"
        f"(Radiomics model, top {TOP_N_SHAP} features by mean |SHAP|)",
        fontsize=12, fontweight="normal", pad=14,
    )
    plt.tight_layout()
    for ext in ("png", "pdf"):
        fig_bee.savefig(
            SHA_DIR / f"shap_beeswarm_{fname_stem}.{ext}",
            dpi=300, bbox_inches="tight", facecolor="white",
        )
    plt.show()
    plt.close(fig_bee)
    print(f"  ✓ Beeswarm saved → shap_beeswarm_{fname_stem}.png / .pdf")

    # -------------------------------------------------------------------------
    # STEP 7 — Bar chart (mean |SHAP| per feature)
    # -------------------------------------------------------------------------
    top_names  = feature_names[top_idx]
    top_shap   = mean_abs_shap[top_idx]
    top_coefs  = coefs[top_idx]

    # Colour bars by sign of LASSO coefficient
    bar_colors = [
        MODEL_COLORS["Combined"] if c > 0
        else MODEL_COLORS["Radiomics"]
        for c in top_coefs
    ]

    fig_bar, ax_bar = plt.subplots(figsize=(9, 7))
    ax_bar.barh(
        range(TOP_N_SHAP), top_shap[::-1],
        color=bar_colors[::-1], edgecolor="white",
        height=0.7,
    )
    ax_bar.set_yticks(range(TOP_N_SHAP))
    ax_bar.set_yticklabels(
        [n.replace("original_", "") for n in top_names[::-1]],
        fontsize=9,
    )
    ax_bar.set_xlabel("Mean |SHAP value|", fontsize=11)
    ax_bar.set_title(
        f"SHAP Feature Importance — {outcome_label}\n"
        f"(Radiomics model, top {TOP_N_SHAP} features)",
        fontsize=12, fontweight="normal",
    )

    # Legend for bar colour coding
    import matplotlib.patches as mpatches
    legend_handles = [
        mpatches.Patch(color=MODEL_COLORS["Combined"],
                       label="Positive coefficient"),
        mpatches.Patch(color=MODEL_COLORS["Radiomics"],
                       label="Negative coefficient"),
    ]
    ax_bar.legend(handles=legend_handles, fontsize=9,
                  loc="lower right", framealpha=0.9)
    ax_bar.grid(axis="x", alpha=0.3, linestyle=":")

    plt.tight_layout()
    for ext in ("png", "pdf"):
        fig_bar.savefig(
            SHA_DIR / f"shap_bar_{fname_stem}.{ext}",
            dpi=300, bbox_inches="tight", facecolor="white",
        )
    plt.show()
    plt.close(fig_bar)
    print(f"  ✓ Bar chart saved → shap_bar_{fname_stem}.png / .pdf")

    return shap_df


# =============================================================================
# RUN — Outcome 1
# =============================================================================
print("=" * 60)
print("SHAP — Outcome 1: Mild vs No HE")
print("=" * 60)
shap_df_o1 = run_shap(
    X_rad         = X_o1_rad,
    y             = y_o1,
    best_Cs       = Cs_o1_rad,
    outcome_label = "Outcome 1 — Mild vs No HE",
    fname_stem    = "outcome1",
)

# =============================================================================
# RUN — Outcome 2
# =============================================================================
print("\n" + "=" * 60)
print("SHAP — Outcome 2: Moderate-Severe vs Mild HE")
print("=" * 60)
shap_df_o2 = run_shap(
    X_rad         = X_o2_rad,
    y             = y_o2,
    best_Cs       = Cs_o2_rad,
    outcome_label = "Outcome 2 — Moderate-Severe vs Mild HE",
    fname_stem    = "outcome2",
)

# =============================================================================
# INLINE SUMMARY — top 5 features per outcome
# =============================================================================
th  = ("padding:5px 12px; text-align:center; font-weight:normal; "
       "border-top:2px solid #111; border-bottom:1px solid #111; "
       "background:#f7f7f7;")
th_l = th.replace("center", "left")
td   = "padding:5px 12px; text-align:center;"
td_l = td.replace("center", "left")

html = """
<div style="font-family:'Helvetica Neue',Arial,sans-serif;
            font-size:12px; max-width:820px; margin-top:12px;">
  <p style="font-size:12px; margin:0 0 8px 0;">
    <b>Top 5 SHAP features — Radiomics model</b>
  </p>"""

for label, shap_df in [
    ("Outcome 1 — Mild vs No HE",        shap_df_o1),
    ("Outcome 2 — Moderate-Severe vs Mild HE", shap_df_o2),
]:
    html += f"""
  <p style="margin:12px 0 4px 0;"><b>{label}</b></p>
  <table style="border-collapse:collapse; width:100%;
                margin-bottom:12px;">
    <thead>
      <tr>
        <th style="{th_l}">Rank</th>
        <th style="{th_l}">Feature</th>
        <th style="{th}">Mean |SHAP|</th>
        <th style="{th}">LASSO coef.</th>
        <th style="{th}">Selected</th>
      </tr>
    </thead>
    <tbody>"""
    for rank, (_, row) in enumerate(shap_df.head(5).iterrows(), 1):
        feat = row["Feature"].replace("original_", "")
        html += f"""
      <tr style="border-bottom:0.5px solid #e8e8e8;">
        <td style="{td_l}">{rank}</td>
        <td style="{td_l}">{feat}</td>
        <td style="{td}">{row['Mean_Abs_SHAP']:.4f}</td>
        <td style="{td}">{row['LASSO_Coef']:+.4f}</td>
        <td style="{td}">{'✓' if row['Selected'] else '—'}</td>
      </tr>"""
    html += """
    </tbody>
  </table>"""

html += "\n</div>"
display(HTML(html))

print(f"\n✅  Cell 8 — SHAP feature importance complete.")

In [ ]:
# =============================================================================
# Cell 9 — AUC grouped bar chart
#           Outcome 1 and Outcome 2 (side by side)
# =============================================================================
# Displays AUC point estimates with 95% bootstrap CI error bars for all
# 3 models across both outcomes in a single grouped figure (Figure 7
# in the manuscript).
#
# Layout:
#   Two groups of 3 bars (one group per outcome), side by side.
#   Each bar represents one model; error bars show 95% bootstrap CI
#   from Cell 5. DeLong significance annotations (vs MELD reference)
#   are displayed above the Radiomics and Combined bars.
#
# Significance notation (DeLong test vs MELD):
#   ***  p < 0.001
#   **   p < 0.01
#   *    p < 0.05
#   ns   not significant
#   —    reference (MELD)
#
# Colours (consistent with all other figures):
#   MELD      → green  #079A02
#   Radiomics → blue   #005DCE
#   Combined  → red    #A60628
#
# Output:
#   outputs/figures/barcharts/auc_barchart.png / .pdf / .svg
#
# TRIPOD-AI items covered:
#   - 10a: discrimination reported with uncertainty
#   - 12 : statistical comparisons annotated
# =============================================================================

def sig_annotation(p_val):
    """
    Return significance string for DeLong p-value.
    Returns empty string for the MELD reference row.
    """
    if p_val is None or (isinstance(p_val, float) and np.isnan(p_val)):
        return "—"
    if p_val < 0.001: return "***"
    if p_val < 0.01:  return "**"
    if p_val < 0.05:  return "*"
    return "ns"


def draw_significance_bracket(ax, x1, x2, y, h, text,
                               color="#333333", fontsize=9.5):
    """
    Draw a significance bracket between two bars.

    Parameters
    ----------
    x1, x2 : float, x-coordinates of the two bars
    y       : float, y-coordinate of the bracket base
    h       : float, height of the bracket arms
    text    : str, annotation text
    """
    ax.plot([x1, x1, x2, x2],
            [y, y + h, y + h, y],
            lw=1.0, color=color)
    ax.text((x1 + x2) / 2, y + h + 0.003,
            text, ha="center", va="bottom",
            fontsize=fontsize, color=color)


# =============================================================================
# EXTRACT METRICS FROM CELL 5 OUTPUT
# =============================================================================
# df_metrics was created in Cell 5 and contains AUC, CI, DeLong p
# for all 6 model × outcome combinations.

outcomes_plot = [
    ("Outcome 1 — Mild vs No HE",        "O1"),
    ("Outcome 2 — Mod-Severe vs Mild HE", "O2"),
]

# Verify df_metrics is available
assert "df_metrics" in dir(), (
    "df_metrics not found — please run Cell 5 before Cell 9."
)

# =============================================================================
# FIGURE
# =============================================================================
fig, ax = plt.subplots(figsize=(11, 7))

bar_width  = 0.22      # width of individual bars
group_gap  = 0.90      # gap between outcome groups
n_models   = 3

# x-positions: two groups, each with 3 bars
group_centres = np.array([0.0, group_gap + n_models * bar_width])
offsets       = np.array([-bar_width, 0.0, bar_width])

all_bar_tops  = []   # used to set y-limit after all bars drawn

for g_idx, (outcome_str, outcome_short) in enumerate(outcomes_plot):

    # Retrieve rows — partial match to handle minor label differences
    df_out = df_metrics[
        df_metrics["Outcome"].str.contains(
            "Outcome 1" if "O1" in outcome_short else "Outcome 2"
        )
    ].reset_index(drop=True)

    group_x = group_centres[g_idx]

    for m_idx, model_label in enumerate(MODEL_LABELS):
        row = df_out[df_out["Model"] == model_label].iloc[0]

        auc    = float(row["AUC"])
        ci_lo  = float(row["AUC_CI_lo"])
        ci_hi  = float(row["AUC_CI_hi"])
        p_val  = row["DeLong_p"]
        color  = MODEL_COLORS[model_label]

        bar_x  = group_x + offsets[m_idx]
        err_lo = auc - ci_lo
        err_hi = ci_hi - auc

        # Bar
        ax.bar(bar_x, auc,
               width      = bar_width * 0.88,
               color      = color,
               edgecolor  = "white",
               linewidth  = 0.8,
               alpha      = 0.88,
               label      = model_label if g_idx == 0 else "_nolegend_",
               zorder     = 3)

        # Error bar (95% bootstrap CI)
        ax.errorbar(bar_x, auc,
                    yerr      = [[err_lo], [err_hi]],
                    fmt       = "none",
                    color     = "#333333",
                    capsize   = 4,
                    capthick  = 1.2,
                    linewidth = 1.2,
                    zorder    = 4)

        # AUC value label inside bar
        ax.text(bar_x, ci_lo - 0.01,
                f"{auc:.3f}",
                ha        = "center",
                va        = "top",
                fontsize  = 8.5,
                color     = "white" if auc > 0.55 else "#333333",
                fontweight= "bold",
                zorder    = 5)

        all_bar_tops.append(ci_hi)

    # -----------------------------------------------------------------
    # Significance annotations (Radiomics vs MELD, Combined vs MELD)
    # -----------------------------------------------------------------
    row_meld  = df_out[df_out["Model"] == "MELD"].iloc[0]
    row_rad   = df_out[df_out["Model"] == "Radiomics"].iloc[0]
    row_combo = df_out[df_out["Model"] == "Combined"].iloc[0]

    x_meld  = group_x + offsets[0]
    x_rad   = group_x + offsets[1]
    x_combo = group_x + offsets[2]

    bracket_base = max(
        float(row_meld["AUC_CI_hi"]),
        float(row_rad["AUC_CI_hi"]),
        float(row_combo["AUC_CI_hi"]),
    ) + 0.025

    # Radiomics vs MELD
    sig_rad = sig_annotation(row_rad["DeLong_p"])
    if sig_rad != "—":
        draw_significance_bracket(
            ax, x_meld, x_rad,
            y    = bracket_base,
            h    = 0.015,
            text = sig_rad,
        )

    # Combined vs MELD (slightly higher bracket)
    sig_combo = sig_annotation(row_combo["DeLong_p"])
    if sig_combo != "—":
        draw_significance_bracket(
            ax, x_meld, x_combo,
            y    = bracket_base + 0.042,
            h    = 0.015,
            text = sig_combo,
        )

# =============================================================================
# AXES FORMATTING
# =============================================================================
# x-tick labels centred on each group
tick_x = [
    group_centres[g] + offsets[1]   # centre bar of each group
    for g in range(len(outcomes_plot))
]
ax.set_xticks(tick_x)
ax.set_xticklabels(
    ["Outcome 1\nMild vs No HE",
     "Outcome 2\nMod-Severe vs Mild HE"],
    fontsize = 11,
)

y_max = max(all_bar_tops) + 0.12
ax.set_ylim(0.40, min(y_max, 1.02))
ax.set_ylabel("AUC (95% Bootstrap CI)", fontsize=12)
ax.set_title(
    "Model Discrimination — AUC with 95% Confidence Intervals",
    fontsize     = 13,
    fontweight   = "bold",
    pad          = 14,
)

# Reference line at AUC = 0.5
ax.axhline(0.5, color="#aaaaaa", linewidth=0.8,
           linestyle="--", zorder=1)
ax.text(ax.get_xlim()[1] * 0.98, 0.502,
        "AUC = 0.5", ha="right", va="bottom",
        fontsize=8, color="#aaaaaa")

ax.grid(axis="y", alpha=0.3, linestyle=":")
ax.legend(
    title      = "Model",
    title_fontsize = 10,
    fontsize   = 10,
    loc        = "lower right",
    frameon    = True,
    shadow     = True,
    framealpha = 0.9,
)

# Significance key
ax.text(
    0.01, 0.01,
    "Significance vs MELD:  * p<0.05   ** p<0.01   *** p<0.001   ns = not significant",
    transform = ax.transAxes,
    fontsize  = 8.5,
    color     = "#555555",
    va        = "bottom",
)

plt.tight_layout()

# =============================================================================
# SAVE
# =============================================================================
for ext in ("png", "pdf", "svg"):
    fig.savefig(
        BAR_DIR / f"auc_barchart.{ext}",
        dpi=300, bbox_inches="tight", facecolor="white",
    )

plt.show()
plt.close(fig)

print(f"  ✓ Saved → {(BAR_DIR / 'auc_barchart')}.png / .pdf / .svg")
print(f"\n✅  Cell 9 — AUC bar chart complete.")

In [ ]:
# =============================================================================
# Cell 10 — Cross-system validation
#            Train on scanner A → Test on scanner B (and vice versa)
# =============================================================================
# Evaluates the technical generalisability of each model across MRI
# acquisition systems using a leave-one-system-out design, as described
# in the manuscript (Cross-System Validation section).
#
# Design:
#   Path 1: Signa 1.5T  → Discovery 3.0T
#   Path 2: Discovery 3.0T → Signa 1.5T
#
# For each path × outcome × model:
#   1. Train on all subjects from the source scanner.
#   2. Apply without harmonisation to all subjects from the target scanner.
#   3. Compute AUC + 95% bootstrap CI (n = 2,000 resamples).
#   4. Permutation test (n = 2,000 iterations) vs null hypothesis AUC = 0.5.
#   5. DeLong test comparing cross-system AUC vs within-scanner OOF AUC
#      on the same test-set subjects.
#   6. Brier score, calibration slope and intercept on test set.
#
# Hyperparameter:
#   Best C = median of outer-fold best C values from Cell 4, per model
#   and outcome, to avoid refitting the inner loop on the source scanner.
#
# Outputs:
#   outputs/tables/cross_system_validation.csv
#   outputs/figures/crosssystem/crosssystem_outcome1.png / .pdf
#   outputs/figures/crosssystem/crosssystem_outcome2.png / .pdf
#   Inline HTML summary table
#
# TRIPOD-AI items covered:
#   - 10d: cross-system technical generalisation
#   - 11 : uncertainty quantification (bootstrap CI)
#   - 12 : permutation test and DeLong test
# =============================================================================

# =============================================================================
# HELPER — Permutation test (AUC vs null hypothesis = 0.5)
# =============================================================================

def permutation_test_auc(y_test, p_test, n_perm=2000, seed=BOOT_SEED):
    """
    Two-sided permutation test: H0: AUC = 0.5 (random classifier).

    Parameters
    ----------
    y_test : np.ndarray, binary outcome on test set
    p_test : np.ndarray, predicted probabilities on test set
    n_perm : int, number of permutation iterations
    seed   : int, random seed

    Returns
    -------
    p_value : float
    """
    rng      = np.random.default_rng(seed)
    obs_auc  = roc_auc_score(y_test, p_test)
    null_auc = np.array([
        roc_auc_score(rng.permutation(y_test), p_test)
        for _ in range(n_perm)
    ])
    # Two-sided: fraction of null AUCs as extreme as observed
    p_val = np.mean(np.abs(null_auc - 0.5) >= np.abs(obs_auc - 0.5))
    return float(p_val)


# =============================================================================
# HELPER — Bootstrap AUC CI
# =============================================================================

def bootstrap_auc_ci(y, p, n_boot=N_BOOT, seed=BOOT_SEED):
    """
    Percentile bootstrap 95% CI for AUC.

    Returns
    -------
    auc    : float, point estimate
    ci_lo  : float
    ci_hi  : float
    """
    rng   = np.random.default_rng(seed)
    aucs  = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        yb, pb = y[idx], p[idx]
        if len(np.unique(yb)) < 2:
            continue
        aucs.append(roc_auc_score(yb, pb))
    aucs = np.array(aucs)
    return (roc_auc_score(y, p),
            float(np.percentile(aucs, 2.5)),
            float(np.percentile(aucs, 97.5)))


# =============================================================================
# HELPER — Calibration metrics (slope, intercept, Brier)
# =============================================================================

def calibration_metrics(y, p):
    """
    Compute Brier score, calibration slope, and intercept
    from logistic regression of y on logit(p̂).
    """
    eps   = 1e-6
    p_c   = np.clip(p, eps, 1 - eps)
    logit = np.log(p_c / (1 - p_c))
    lr    = LogisticRegression(penalty=None, solver="lbfgs", max_iter=500)
    lr.fit(logit.reshape(-1, 1), y)
    return {
        "Brier":         float(brier_score_loss(y, p_c)),
        "Cal_Slope":     float(lr.coef_[0][0]),
        "Cal_Intercept": float(lr.intercept_[0]),
    }


# =============================================================================
# MAIN — Cross-system validation loop
# =============================================================================

# Best C values from Cell 4 — one list per model × outcome
BEST_C_MAP = {
    ("O1", "MELD"):      np.median(Cs_o1_meld),
    ("O1", "Radiomics"): np.median(Cs_o1_rad),
    ("O1", "Combined"):  np.median(Cs_o1_combo),
    ("O2", "MELD"):      np.median(Cs_o2_meld),
    ("O2", "Radiomics"): np.median(Cs_o2_rad),
    ("O2", "Combined"):  np.median(Cs_o2_combo),
}

# Feature matrices and labels per outcome
OUTCOME_DATA = {
    "O1": {
        "label":   "Outcome 1 — Mild vs No HE",
        "df":       df_o1,
        "y":        y_o1,
        "scanner":  scanner_o1,
        "X_meld":   X_o1_meld,
        "X_rad":    X_o1_rad,
        "X_combo":  X_o1_combo,
        "p_oof": {
            "MELD":      p_o1_meld,
            "Radiomics": p_o1_rad,
            "Combined":  p_o1_combo,
        },
    },
    "O2": {
        "label":   "Outcome 2 — Moderate-Severe vs Mild HE",
        "df":       df_o2,
        "y":        y_o2,
        "scanner":  scanner_o2,
        "X_meld":   X_o2_meld,
        "X_rad":    X_o2_rad,
        "X_combo":  X_o2_combo,
        "p_oof": {
            "MELD":      p_o2_meld,
            "Radiomics": p_o2_rad,
            "Combined":  p_o2_combo,
        },
    },
}

X_MAP = {
    "MELD":      "X_meld",
    "Radiomics": "X_rad",
    "Combined":  "X_combo",
}

rows_cs = []

for outcome_key, odata in OUTCOME_DATA.items():
    print(f"\n{'='*60}")
    print(f"{odata['label']}")
    print(f"{'='*60}")

    y_all       = odata["y"]
    scanner_all = odata["scanner"]

    for src_scanner, tgt_scanner, path_label in CROSS_PATHS:
        print(f"\n  Path: {path_label}")

        src_mask = scanner_all == f"{src_scanner.replace('Signa','1.5T').replace('Discovery','3.0T')}"
        tgt_mask = scanner_all == f"{tgt_scanner.replace('Signa','1.5T').replace('Discovery','3.0T')}"

        # Fallback: match raw scanner labels
        if src_mask.sum() == 0:
            src_mask = np.array([s == src_scanner for s in
                                 odata["df"][COL_MR_SYSTEM].values])
            tgt_mask = np.array([s == tgt_scanner for s in
                                 odata["df"][COL_MR_SYSTEM].values])

        y_train = y_all[src_mask]
        y_test  = y_all[tgt_mask]

        if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
            print(f"    ⚠️  Skipped — degenerate split "
                  f"(train n={src_mask.sum()}, test n={tgt_mask.sum()})")
            continue

        for model_label in MODEL_LABELS:
            X_all   = odata[X_MAP[model_label]]
            X_train = X_all[src_mask]
            X_test  = X_all[tgt_mask]
            best_C  = BEST_C_MAP[(outcome_key, model_label)]

            # Train on source scanner
            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("clf",    LogisticRegression(
                    penalty      = "l1",
                    C            = best_C,
                    solver       = "liblinear",
                    class_weight = "balanced",
                    max_iter     = 2000,
                    random_state = RANDOM_STATE,
                )),
            ])
            pipe.fit(X_train, y_train)
            p_test = pipe.predict_proba(X_test)[:, 1]

            # AUC + bootstrap CI
            auc, ci_lo, ci_hi = bootstrap_auc_ci(y_test, p_test)

            # Permutation test vs AUC = 0.5
            p_perm = permutation_test_auc(y_test, p_test)

            # DeLong test: cross-system AUC vs within-scanner OOF AUC
            # (on the same test subjects)
            p_oof_test = odata["p_oof"][model_label][tgt_mask]
            _, p_delong_cs = delong_test(y_test, p_test, p_oof_test)

            # Calibration metrics
            cal = calibration_metrics(y_test, p_test)

            # Significance
            if   p_perm < 0.001: sig_perm = "***"
            elif p_perm < 0.01:  sig_perm = "**"
            elif p_perm < 0.05:  sig_perm = "*"
            else:                sig_perm = "ns"

            rows_cs.append({
                "Outcome":     odata["label"],
                "Path":        path_label,
                "Model":       model_label,
                "N_train":     int(src_mask.sum()),
                "N_test":      int(tgt_mask.sum()),
                "AUC":         auc,
                "AUC_CI_lo":   ci_lo,
                "AUC_CI_hi":   ci_hi,
                "Brier":       cal["Brier"],
                "Cal_Slope":   cal["Cal_Slope"],
                "Cal_Int":     cal["Cal_Intercept"],
                "Perm_p":      p_perm,
                "Sig_perm":    sig_perm,
                "DeLong_cs_p": p_delong_cs,
            })

            print(f"    {model_label:<12} "
                  f"AUC {auc:.3f} ({ci_lo:.3f}–{ci_hi:.3f})  "
                  f"Brier {cal['Brier']:.3f}  "
                  f"Perm p: {p_perm:.3f} {sig_perm}  "
                  f"DeLong (vs OOF): {p_delong_cs:.3f}")

# =============================================================================
# SAVE CSV
# =============================================================================
df_cs = pd.DataFrame(rows_cs)
df_cs.to_csv(TAB_DIR / "cross_system_validation.csv", index=False)
print(f"\n  ✓ Saved → {(TAB_DIR / 'cross_system_validation.csv').resolve()}")

# =============================================================================
# FIGURE — AUC heatmap per outcome
# =============================================================================

def plot_crosssystem_figure(outcome_label, df_cs_out, fname):
    """
    Grouped bar chart of cross-system AUCs with CI error bars,
    one panel per migration path, for a single outcome.
    """
    paths = df_cs_out["Path"].unique()
    n_paths = len(paths)

    fig, axes = plt.subplots(1, n_paths,
                             figsize=(6 * n_paths, 6),
                             sharey=True)
    if n_paths == 1:
        axes = [axes]

    fig.suptitle(f"Cross-System Validation — {outcome_label}",
                 fontsize=12, fontweight="bold", y=1.01)

    bar_width = 0.22
    offsets   = np.array([-bar_width, 0.0, bar_width])

    for ax, path in zip(axes, paths):
        df_path = df_cs_out[df_cs_out["Path"] == path]

        for m_idx, model_label in enumerate(MODEL_LABELS):
            row    = df_path[df_path["Model"] == model_label]
            if row.empty:
                continue
            row    = row.iloc[0]
            auc    = float(row["AUC"])
            ci_lo  = float(row["AUC_CI_lo"])
            ci_hi  = float(row["AUC_CI_hi"])
            color  = MODEL_COLORS[model_label]
            bar_x  = offsets[m_idx]

            ax.bar(bar_x, auc,
                   width     = bar_width * 0.88,
                   color     = color,
                   edgecolor = "white",
                   alpha     = 0.88,
                   zorder    = 3)
            ax.errorbar(bar_x, auc,
                        yerr      = [[auc - ci_lo], [ci_hi - auc]],
                        fmt       = "none",
                        color     = "#333333",
                        capsize   = 4,
                        capthick  = 1.2,
                        linewidth = 1.2,
                        zorder    = 4)
            ax.text(bar_x, ci_lo - 0.012,
                    f"{auc:.3f}",
                    ha        = "center",
                    va        = "top",
                    fontsize  = 8.5,
                    color     = "white",
                    fontweight= "bold",
                    zorder    = 5)

            # Permutation significance above bar
            sig = row["Sig_perm"]
            ax.text(bar_x, ci_hi + 0.012,
                    sig,
                    ha       = "center",
                    va       = "bottom",
                    fontsize = 10,
                    color    = color)

        ax.set_xticks(offsets)
        ax.set_xticklabels(MODEL_LABELS, fontsize=10)
        ax.set_title(path, fontsize=11, fontweight="normal")
        ax.axhline(0.5, color="#aaaaaa", linewidth=0.8,
                   linestyle="--", zorder=1)
        ax.set_ylim(0.35, 1.05)
        ax.set_ylabel("AUC (95% Bootstrap CI)", fontsize=10)
        ax.grid(axis="y", alpha=0.3, linestyle=":")

        n_train = int(df_path["N_train"].iloc[0])
        n_test  = int(df_path["N_test"].iloc[0])
        ax.text(0.98, 0.02,
                f"Train n={n_train}  |  Test n={n_test}",
                transform = ax.transAxes,
                ha        = "right",
                va        = "bottom",
                fontsize  = 8,
                color     = "#555555")

    plt.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(CRS_DIR / f"{fname}.{ext}",
                    dpi=300, bbox_inches="tight",
                    facecolor="white")
    plt.show()
    plt.close(fig)
    print(f"  ✓ Saved → {(CRS_DIR / fname)}.png / .pdf")


print("\n▶  Cross-system figures")
for outcome_key, odata in OUTCOME_DATA.items():
    df_cs_out = df_cs[df_cs["Outcome"] == odata["label"]]
    if df_cs_out.empty:
        continue
    fname = f"crosssystem_{'outcome1' if outcome_key == 'O1' else 'outcome2'}"
    plot_crosssystem_figure(odata["label"], df_cs_out, fname)

# =============================================================================
# INLINE HTML SUMMARY TABLE
# =============================================================================
th  = ("padding:5px 12px; text-align:center; font-weight:normal; "
       "border-top:2px solid #111; border-bottom:1px solid #111; "
       "background:#f7f7f7;")
th_l = th.replace("center", "left")
td   = "padding:5px 12px; text-align:center;"
td_l = td.replace("center", "left")

html = """
<div style="font-family:'Helvetica Neue',Arial,sans-serif;
            font-size:12px; max-width:980px; margin-top:12px;">
  <p style="font-size:12px; margin:0 0 8px 0;">
    <b>Cross-System Validation Summary (Supplementary Table ST3)</b>
  </p>"""

for outcome_label in df_cs["Outcome"].unique():
    df_out = df_cs[df_cs["Outcome"] == outcome_label]
    html += f"""
  <p style="margin:12px 0 4px 0;"><b>{outcome_label}</b></p>
  <table style="border-collapse:collapse; width:100%;
                margin-bottom:16px;">
    <thead>
      <tr>
        <th style="{th_l}">Path</th>
        <th style="{th_l}">Model</th>
        <th style="{th}">N train / test</th>
        <th style="{th}">AUC (95% CI)</th>
        <th style="{th}">Brier</th>
        <th style="{th}">Cal. Slope</th>
        <th style="{th}">Cal. Int.</th>
        <th style="{th}">Perm. <i>p</i></th>
        <th style="{th}">DeLong vs OOF</th>
      </tr>
    </thead>
    <tbody>"""

    for path in df_out["Path"].unique():
        df_path = df_out[df_out["Path"] == path]
        for m_idx, model_label in enumerate(MODEL_LABELS):
            row = df_path[df_path["Model"] == model_label]
            if row.empty:
                continue
            row   = row.iloc[0]
            color = f"color:{MODEL_COLORS[model_label]};"
            bold_o = "<b>" if model_label == "Combined" else ""
            bold_c = "</b>" if model_label == "Combined" else ""
            path_cell = (f'<td style="{td_l}" rowspan="3">{path}</td>'
                         if m_idx == 0 else "")

            def p_str(p):
                if np.isnan(p): return "—"
                if p < 0.001:   return "< 0.001"
                if p < 0.01:    return "< 0.01"
                return f"{p:.3f}"

            html += f"""
      <tr style="border-bottom:0.5px solid #e8e8e8;">
        {path_cell}
        <td style="{td_l}{color}">{bold_o}{model_label}{bold_c}</td>
        <td style="{td}">{int(row.N_train)} / {int(row.N_test)}</td>
        <td style="{td}">{bold_o}{row.AUC:.3f} ({row.AUC_CI_lo:.3f}–{row.AUC_CI_hi:.3f}){bold_c}</td>
        <td style="{td}">{row.Brier:.3f}</td>
        <td style="{td}">{row.Cal_Slope:.3f}</td>
        <td style="{td}">{row.Cal_Int:.3f}</td>
        <td style="{td}">{p_str(row.Perm_p)} {row.Sig_perm}</td>
        <td style="{td}">{p_str(row.DeLong_cs_p)}</td>
      </tr>"""

    html += f"""
    </tbody>
    <tfoot>
      <tr style="border-top:2px solid #111;">
        <td colspan="9"
            style="padding:4px 12px; font-size:11px; color:#555;">
          AUC = area under ROC curve; CI = 95% bootstrap CI
          (n = {N_BOOT} resamples); Perm. <i>p</i> = permutation test
          vs H₀: AUC = 0.5 (n = 2,000 iterations); DeLong vs OOF =
          comparison of cross-system AUC with within-scanner OOF AUC
          on identical test subjects.
          * p&lt;0.05, ** p&lt;0.01, *** p&lt;0.001, ns = not significant.
        </td>
      </tr>
    </tfoot>
  </table>"""

html += "\n</div>"
display(HTML(html))

print(f"\n✅  Cell 10 — Cross-system validation complete.")

In [ ]:
# =============================================================================
# Cell 11 — Clinical nomograms
#            Outcome 1 and Outcome 2
# =============================================================================
# Produces publication-ready clinical nomograms integrating MELD score
# and MRI-derived Radiomic Signature for individual risk estimation,
# as described in the Statistical Analysis section of the manuscript.
#
# Methodological notes:
#
# 1. Final model refit:
#    The nomogram is based on an unpenalized logistic regression
#    (penalty=None) refitted on the full dataset using the optimal
#    regularisation parameter identified during nested CV (Cell 4).
#    Coefficients: logit(P) = b0 + b_meld × MELD + b_rad × RadScore
#
# 2. RadScore (leakage-free):
#    RadScore = logit(p_oof_rad) — derived from OOF predictions (Cell 4).
#    This ensures the radiomic contribution is estimated without data
#    leakage; the same OOF RadScore is used in the nomogram as in the
#    performance metrics (Cell 5).
#
# 3. Points mapping (Iasonos et al., J Clin Oncol 2008):
#    Each predictor axis spans a physical width proportional to its
#    LP contribution:
#      delta_i = |b_i| × (max_i − min_i)
#      w_i     = (delta_i / max(delta)) × W
#    A minimum axis width of 30% of W is enforced so that predictors
#    with near-zero coefficients (e.g. MELD for Outcome 2, where
#    AUC ≈ 0.500) remain readable. This is standard practice in
#    clinical nomogram design and does not affect model coefficients.
#
# 4. Visual style:
#    - Vertical dashed reference grid
#    - Gaussian KDE density cloud above MELD and RadScore axes
#    - Axis labels in bold; Probability axis in red (#A60628)
#    - Footnote usage instructions
#
# Outputs:
#   outputs/figures/nomograms/nomogram_outcome1.png / .pdf
#   outputs/figures/nomograms/nomogram_outcome2.png / .pdf
#   outputs/tables/nomogram_coefficients.csv
#   outputs/tables/nomogram_data_outcome1.csv
#   outputs/tables/nomogram_data_outcome2.csv
#   Inline HTML legend (Hepatology style)
#
# TRIPOD-AI items covered:
#   - 13b: presentation tool for individual prediction (nomogram)
#   - 15 : coefficients and nomogram data saved for reproducibility
# =============================================================================

# =============================================================================
# CONSTANTS
# =============================================================================
PROB_TICKS = [0.01, 0.05, 0.10, 0.20, 0.30,
              0.40, 0.50, 0.60, 0.70, 0.80,
              0.90, 0.95, 0.99]

LP_MIN = np.log(0.01 / 0.99)   # −4.5951
LP_MAX = np.log(0.99 / 0.01)   # +4.5951
LP_RNG = LP_MAX - LP_MIN        #  9.1902

LEFT   = 20.0    # x start of all axes (data units)
RIGHT  = 105.0   # x end   of all axes
W      = RIGHT - LEFT
LBL_X  = 18.5   # x position of row labels

# Minimum axis width as fraction of W — ensures readability when a
# predictor coefficient is near zero (e.g. MELD for Outcome 2)
W_MIN_FRAC = 0.30

MELD_TICKS = np.arange(0,  41, 5)
RAD_TICKS  = np.arange(-5,  6,  1)
TICK_H     = 0.25
FONTSIZE   = 10.5
FONTSIZE_S = 8.5


# =============================================================================
# STEP 1 — Build nomogram datasets
# =============================================================================

def make_nomo_dataset(df_sub, y_col, p_rad_oof,
                      outcome_col_name, csv_path):
    """
    Build nomogram input DataFrame:
      - MELD on original scale
      - RadScore = logit(p_oof_rad)  [leakage-free]
      - Binary outcome
    """
    eps       = 1e-6
    p_clip    = np.clip(np.asarray(p_rad_oof, float), eps, 1 - eps)
    rad_score = np.log(p_clip / (1 - p_clip))

    df_nomo = pd.DataFrame({
        outcome_col_name: df_sub[y_col].values.astype(int),
        "MELD":           df_sub[COL_MELD].values.astype(float),
        "RadScore":       rad_score,
    })
    df_nomo.to_csv(csv_path, index=False)
    print(f"  Dataset saved → {csv_path.name}")
    print(f"  n = {len(df_nomo)}  |  "
          f"RadScore [{rad_score.min():.2f}, {rad_score.max():.2f}]")
    return df_nomo


print("▶  Building nomogram datasets")
df_nomo_o1 = make_nomo_dataset(
    df_sub           = df_o1,
    y_col            = COL_HE_MILD,
    p_rad_oof        = p_o1_rad,
    outcome_col_name = "HE_Mild",
    csv_path         = TAB_DIR / "nomogram_data_outcome1.csv",
)
df_nomo_o2 = make_nomo_dataset(
    df_sub           = df_o2,
    y_col            = COL_HE_MODSEV,
    p_rad_oof        = p_o2_rad,
    outcome_col_name = "HE_ModSev",
    csv_path         = TAB_DIR / "nomogram_data_outcome2.csv",
)


# =============================================================================
# STEP 2 — Fit nomogram logistic models (unpenalized, full dataset)
# =============================================================================

def fit_nomo_model(df_nomo, outcome_col):
    """
    Unpenalized logistic regression:
      logit(P) = b0 + b_meld × MELD + b_rad × RadScore
    Refitted on the full dataset using optimal C from nested CV.
    """
    X  = df_nomo[["MELD", "RadScore"]].values
    y  = df_nomo[outcome_col].values.astype(int)
    lr = LogisticRegression(
        penalty  = None,
        solver   = "lbfgs",
        max_iter = 2000,
    )
    lr.fit(X, y)
    b0     = float(lr.intercept_[0])
    b_meld = float(lr.coef_[0][0])
    b_rad  = float(lr.coef_[0][1])
    print(f"  b0 = {b0:.4f}  |  "
          f"b_meld = {b_meld:.4f}  |  "
          f"b_rad = {b_rad:.4f}")
    return b0, b_meld, b_rad


print("\n▶  Fitting nomogram models")
b0_o1, b_meld_o1, b_rad_o1 = fit_nomo_model(df_nomo_o1, "HE_Mild")
b0_o2, b_meld_o2, b_rad_o2 = fit_nomo_model(df_nomo_o2, "HE_ModSev")

# Save coefficients
coef_df = pd.DataFrame([
    {"Outcome": "Outcome 1 — Mild vs No HE",
     "b0": b0_o1, "b_meld": b_meld_o1, "b_rad": b_rad_o1},
    {"Outcome": "Outcome 2 — Mod-Sev vs Mild HE",
     "b0": b0_o2, "b_meld": b_meld_o2, "b_rad": b_rad_o2},
])
coef_df.to_csv(TAB_DIR / "nomogram_coefficients.csv", index=False)
print(f"\n  ✓ Coefficients saved → nomogram_coefficients.csv")


# =============================================================================
# STEP 3 — Axis mapping (Iasonos et al. 2008 + minimum width floor)
# =============================================================================

def make_iasonos_mappers(b_meld, b_rad,
                         meld_lo=0, meld_hi=40,
                         rad_lo=-5, rad_hi=5):
    """
    Build per-predictor axis mappers following Iasonos et al. 2008,
    with a minimum axis width floor of W_MIN_FRAC × W to ensure
    readability when a coefficient is near zero.

    Steps:
      1. delta_i = |b_i| × (hi_i − lo_i)
      2. w_i     = (delta_i / max(delta)) × W
      3. w_i     = max(w_i, W_MIN_FRAC × W)   ← floor
      4. mapper  : x = LEFT + (value − lo_i) / (hi_i − lo_i) × w_i

    All axes are left-anchored at LEFT.

    Returns
    -------
    meld_mapper : callable
    rad_mapper  : callable
    w_meld      : float
    w_rad       : float
    """
    delta_meld = abs(b_meld) * (meld_hi - meld_lo)
    delta_rad  = abs(b_rad)  * (rad_hi  - rad_lo)
    max_delta  = max(delta_meld, delta_rad)

    if max_delta == 0:
        max_delta = 1.0

    w_min = W_MIN_FRAC * W

    w_meld = max((delta_meld / max_delta) * W, w_min)
    w_rad  = max((delta_rad  / max_delta) * W, w_min)

    def meld_mapper(value):
        return LEFT + (value - meld_lo) / (meld_hi - meld_lo) * w_meld

    def rad_mapper(value):
        return LEFT + (value - rad_lo) / (rad_hi - rad_lo) * w_rad

    return meld_mapper, rad_mapper, w_meld, w_rad


def pts_to_x(pts):
    """Points (0–100) → x coordinate (full width)."""
    return LEFT + pts / 100.0 * W


def prob_to_x(prob):
    """Probability → x coordinate via logit scale (full width)."""
    lp = np.log(prob / (1 - prob))
    return LEFT + (lp - LP_MIN) / LP_RNG * W


# =============================================================================
# STEP 4 — Drawing utilities
# =============================================================================

def draw_row_with_kde(ax, row_y, row_label, ticks, mapper,
                      tick_fmt=None, color="black",
                      label_fontsize=None,
                      kde_data=None, kde_range=None):
    """
    Draw one nomogram axis row with optional KDE cloud.
    KDE x-coordinates are derived from mapper() to ensure alignment.
    """
    lfs = label_fontsize or FONTSIZE

    # Row label
    ax.text(LBL_X, row_y, row_label,
            ha="right", va="center",
            fontsize=lfs, fontweight="bold", color=color)

    # Collect valid ticks (within axis bounds)
    valid = []
    for t in ticks:
        x = mapper(t)
        if LEFT - 0.5 <= x <= RIGHT + 0.5:
            valid.append((t, x))

    if not valid:
        return

    # Axis line
    ax.plot([valid[0][1], valid[-1][1]], [row_y, row_y],
            color=color, linewidth=1.8,
            solid_capstyle="butt", zorder=2)

    # Ticks and labels
    for t, x in valid:
        ax.plot([x, x], [row_y - TICK_H, row_y + TICK_H],
                color=color, linewidth=1.5, zorder=3)
        if tick_fmt is not None:
            lbl = tick_fmt(t)
        elif float(t) == int(t):
            lbl = str(int(t))
        else:
            lbl = f"{t:.1f}"
        ax.text(x, row_y - TICK_H - 0.18, lbl,
                ha="center", va="top",
                fontsize=FONTSIZE_S, color=color)

    # KDE density cloud — x_plot from mapper() for correct alignment
    if kde_data is not None and len(kde_data) > 3:
        try:
            kde    = gaussian_kde(kde_data)
            lo     = kde_range[0] if kde_range is not None else float(kde_data.min())
            hi     = kde_range[1] if kde_range is not None else float(kde_data.max())
            x_eval = np.linspace(lo, hi, 150)
            x_plot = np.array([mapper(v) for v in x_eval])
            mask   = (x_plot >= LEFT) & (x_plot <= RIGHT)
            density = (kde(x_eval) / kde(x_eval).max()) * 0.38
            ax.fill_between(x_plot[mask],
                            row_y, row_y + density[mask],
                            color="gray", alpha=0.13, zorder=1)
        except Exception:
            pass


# =============================================================================
# STEP 5 — Main nomogram drawing function
# =============================================================================

def draw_nomogram(b0, b_meld, b_rad,
                  data_meld, data_rad,
                  title, fname):
    """
    Draw publication-ready nomogram.

    Row layout (top → bottom):
      Row 1 : Points         (0–100, full width)
      Row 2 : MELD           (width ∝ LP contribution, min 30% W)
      Row 3 : Radiomic Score (width ∝ LP contribution, min 30% W)
      Row 4 : Total Points   (0–100, full width)
      Row 5 : Probability    (0.01–0.99, red, logit scale, full width)
    """
    meld_mapper, rad_mapper, w_meld, w_rad = make_iasonos_mappers(
        b_meld, b_rad,
        meld_lo=0, meld_hi=40,
        rad_lo=-5, rad_hi=5,
    )

    print(f"  Iasonos widths — MELD: {w_meld:.1f}  Rad: {w_rad:.1f}  "
          f"(full W={W:.1f},  floor={W_MIN_FRAC*W:.1f})")

    fig, ax = plt.subplots(figsize=(11, 8.5))
    fig.patch.set_facecolor("white")
    ax.set_xlim(0, 115)
    ax.set_ylim(0, 13)
    ax.axis("off")

    ROW = {
        "points":      11.5,
        "meld":         9.0,
        "rad":          6.5,
        "total":        4.0,
        "probability":  1.5,
    }

    # Vertical dashed reference grid (aligned to Points / full width)
    for v in range(0, 101, 10):
        xd = LEFT + v / 100 * W
        ax.plot([xd, xd],
                [ROW["probability"] - 0.4, ROW["points"] + 0.3],
                color="#dddddd", lw=0.7, ls="--", alpha=0.5, zorder=0)

    pts_ticks = np.arange(0, 101, 10)

    # Row 1: Points
    draw_row_with_kde(ax, ROW["points"], "Points",
                      pts_ticks, pts_to_x)

    # Row 2: MELD
    draw_row_with_kde(ax, ROW["meld"], "MELD Score",
                      MELD_TICKS, meld_mapper,
                      kde_data  = data_meld,
                      kde_range = (0, 40))

    # Row 3: Radiomic Score
    draw_row_with_kde(ax, ROW["rad"], "Rad\nSignature",
                      RAD_TICKS, rad_mapper,
                      tick_fmt       = lambda v: f"{int(v):+d}",
                      label_fontsize = 9.5,
                      kde_data       = data_rad,
                      kde_range      = (-5, 5))

    # Row 4: Total Points
    draw_row_with_kde(ax, ROW["total"], "Total\nPoints",
                      pts_ticks, pts_to_x,
                      label_fontsize = 9.5)

    # Row 5: Probability (red, logit scale)
    prob_color = "#A60628"
    ax.text(LBL_X, ROW["probability"], "Probability\nof HE",
            ha="right", va="center",
            fontsize=FONTSIZE, fontweight="bold", color=prob_color)

    x_lo = prob_to_x(PROB_TICKS[0])
    x_hi = prob_to_x(PROB_TICKS[-1])
    ax.plot([x_lo, x_hi], [ROW["probability"], ROW["probability"]],
            color=prob_color, linewidth=1.8,
            solid_capstyle="butt", zorder=2)

    for prob in PROB_TICKS:
        x = prob_to_x(prob)
        ax.plot([x, x],
                [ROW["probability"] - TICK_H,
                 ROW["probability"] + TICK_H],
                color=prob_color, linewidth=1.5, zorder=3)
        lbl = (f"{int(prob*100)}%"
               if prob not in (0.01, 0.05, 0.95, 0.99)
               else f"{prob:.2f}")
        ax.text(x, ROW["probability"] - TICK_H - 0.18,
                lbl, ha="center", va="top",
                fontsize=FONTSIZE_S, fontweight="bold",
                color=prob_color)

    # Title
    ax.set_title(title, fontsize=14, fontweight="bold",
                 color="#222222", pad=10)

    # Usage footnote
    ax.text(
        (LEFT + RIGHT) / 2, 0.3,
        "To use: draw a vertical line from each predictor axis "
        "to the Points axis and sum the values. "
        "Draw a vertical line from Total Points to the "
        "Probability axis to read the estimated individual risk.",
        ha="center", va="bottom",
        fontsize=7.5, color="#555555", style="italic",
    )

    plt.tight_layout(pad=2.0)

    for ext in ("png", "pdf"):
        fig.savefig(NOM_DIR / f"{fname}.{ext}",
                    dpi=300, bbox_inches="tight",
                    facecolor="white")

    plt.show()
    plt.close(fig)
    print(f"  ✓ Saved → {(NOM_DIR / fname)}.png / .pdf")


# =============================================================================
# STEP 6 — Inline figure legend (Hepatology style)
# =============================================================================

def print_nomogram_legend(outcome_label, fname,
                          b0, b_meld, b_rad,
                          df_nomo, outcome_col):
    """
    Inline HTML legend with coefficient table and apparent AUC.
    """
    X   = df_nomo[["MELD", "RadScore"]].values
    y   = df_nomo[outcome_col].values.astype(int)
    lr  = LogisticRegression(penalty=None, solver="lbfgs", max_iter=2000)
    lr.fit(X, y)
    auc_app = roc_auc_score(y, lr.predict_proba(X)[:, 1])

    th  = ("padding:5px 12px; text-align:center; font-weight:normal; "
           "border-top:2px solid #111; border-bottom:1px solid #111; "
           "background:#f7f7f7;")
    th_l = th.replace("center", "left")
    td   = "padding:5px 12px; text-align:center;"
    td_l = td.replace("center", "left")

    html = f"""
    <div style="font-family:'Helvetica Neue',Arial,sans-serif;
                font-size:12px; max-width:820px;
                margin-bottom:28px; margin-top:8px;">

      <p style="font-size:12px; margin:0 0 6px 0;">
        <b>Figure legend — Nomogram ({outcome_label})</b>
      </p>

      <p style="margin:0 0 10px 0; color:#333; line-height:1.6;">
        Clinical-radiomic nomogram integrating the MELD score and the
        MRI-derived Radiomic Signature for the estimation of individual
        probability of {outcome_label.lower()}.
        The Radiomic Score (RadScore) is derived from out-of-fold
        cross-validation predictions (logit-transformed) to ensure
        calibration integrity and avoid data leakage (TRIPOD-AI item 10a).
        Each predictor axis spans a physical width proportional to its
        contribution to the linear predictor (delta = |b| × predictor range),
        with the dominant predictor assigned the full axis width and a
        minimum readable width enforced for all axes
        (Iasonos et al., <i>J Clin Oncol</i> 2008;26:1364–1370).
        Gray density clouds above the MELD and Radiomic Score axes
        represent the empirical distribution of predictor values in
        the cohort (Gaussian KDE). The probability axis (red) is
        rendered on a logit scale anchored at 0.01–0.99.
        Apparent AUC of the nomogram model on full data:
        <b>{auc_app:.3f}</b> (optimistic — validated performance
        estimates are reported in Tables 2–3).
      </p>

      <p style="margin:0 0 6px 0;"><b>Model coefficients</b></p>
      <table style="border-collapse:collapse; width:auto;
                    margin-bottom:12px;">
        <thead>
          <tr>
            <th style="{th_l}">Parameter</th>
            <th style="{th}">Value</th>
            <th style="{th}">Interpretation</th>
          </tr>
        </thead>
        <tbody>
          <tr style="border-bottom:0.5px solid #e8e8e8;">
            <td style="{td_l}">Intercept (b₀)</td>
            <td style="{td}">{b0:.4f}</td>
            <td style="{td}">Baseline log-odds</td>
          </tr>
          <tr style="border-bottom:0.5px solid #e8e8e8;">
            <td style="{td_l}">b_MELD</td>
            <td style="{td}">{b_meld:.4f}</td>
            <td style="{td}">Log-odds change per 1-unit MELD increase</td>
          </tr>
          <tr style="border-bottom:0.5px solid #e8e8e8;">
            <td style="{td_l}">b_RadScore</td>
            <td style="{td}">{b_rad:.4f}</td>
            <td style="{td}">Log-odds change per 1-unit RadScore increase</td>
          </tr>
          <tr style="border-top:2px solid #111;">
            <td style="{td_l}">Apparent AUC</td>
            <td style="{td}">{auc_app:.3f}</td>
            <td style="{td}">Full-data fit — optimistic (see Tables 2–3)</td>
          </tr>
        </tbody>
      </table>

      <p style="margin:0 0 6px 0;"><b>How to use this nomogram</b></p>
      <ol style="color:#333; line-height:1.8; margin:0 0 8px 0;">
        <li>Locate the patient&#x2019;s MELD score on the MELD Score
            axis and draw a vertical line to the Points axis.
            Note the corresponding points value.</li>
        <li>Locate the patient&#x2019;s Radiomic Score on the Rad
            Signature axis and draw a vertical line to the Points axis.
            Note the corresponding points value.</li>
        <li>Sum the two points values to obtain the Total Points.</li>
        <li>Locate the Total Points value on the Total Points axis
            and draw a vertical line down to the Probability axis.
            Read the estimated individual probability of
            {outcome_label.lower()}.</li>
      </ol>

      <p style="font-size:11px; color:#555; margin:4px 0 0 0;">
        Saved: <code>{fname}.png / .pdf</code>
      </p>
    </div>"""

    display(HTML(html))


# =============================================================================
# RENDER — Outcome 1
# =============================================================================
print("\n▶  Nomogram — Outcome 1 (Mild vs No HE)")
draw_nomogram(
    b0        = b0_o1,
    b_meld    = b_meld_o1,
    b_rad     = b_rad_o1,
    data_meld = df_nomo_o1["MELD"].values,
    data_rad  = df_nomo_o1["RadScore"].values,
    title     = ("Nomogram — Outcome 1: "
                 "Prediction of Mild Hepatic Encephalopathy\n"
                 "(MELD Score + Radiomic Signature)"),
    fname     = "nomogram_outcome1",
)
print_nomogram_legend(
    outcome_label = "Outcome 1 — Mild vs No HE",
    fname         = "nomogram_outcome1",
    b0            = b0_o1,
    b_meld        = b_meld_o1,
    b_rad         = b_rad_o1,
    df_nomo       = df_nomo_o1,
    outcome_col   = "HE_Mild",
)

# =============================================================================
# RENDER — Outcome 2
# =============================================================================
print("\n▶  Nomogram — Outcome 2 (Moderate-Severe vs Mild HE)")
draw_nomogram(
    b0        = b0_o2,
    b_meld    = b_meld_o2,
    b_rad     = b_rad_o2,
    data_meld = df_nomo_o2["MELD"].values,
    data_rad  = df_nomo_o2["RadScore"].values,
    title     = ("Nomogram — Outcome 2: "
                 "Differentiation of Moderate-to-Severe "
                 "vs Mild Hepatic Encephalopathy\n"
                 "(MELD Score + Radiomic Signature)"),
    fname     = "nomogram_outcome2",
)
print_nomogram_legend(
    outcome_label = "Outcome 2 — Moderate-Severe vs Mild HE",
    fname         = "nomogram_outcome2",
    b0            = b0_o2,
    b_meld        = b_meld_o2,
    b_rad         = b_rad_o2,
    df_nomo       = df_nomo_o2,
    outcome_col   = "HE_ModSev",
)

print(f"\n✅  Cell 11 — Nomograms complete.")

In [ ]:
# =============================================================================
# Cell 11 — Nomograms
#            Outcome 1 and Outcome 2 — side-by-side two-panel figure
# =============================================================================
# Produces a publication-ready two-panel figure (Figure 2B):
#   Left panel  — Outcome 1: Mild HE vs No HE
#   Right panel — Outcome 2: Moderate-to-Severe HE vs Mild HE
#
# Layout per panel (classic 5-row nomogram):
#   Row 1 : Points       (0–100, full width)
#   Row 2 : MELD Score   (0–40,  full width, KDE)
#   Row 3 : RadScore     (−5/+5, full width, KDE)
#   Row 4 : Total Points (0–100, full width)
#   Row 5 : Probability  (0.01–0.99, red, logit scale)
#
# Both predictor axes always span full width LEFT → RIGHT.
# The Points axis (0–100) encodes the sum of contributions via
# the logit-scale linear predictor mapping (Iasonos et al. 2008).
#
# Methodological notes:
#   - RadScore = logit(p_oof_rad): leakage-free (Cell 4)
#   - Unpenalized logistic regression refitted on full dataset
#   - KDE clouds aligned to axis mapper (v1 fix)
#   - Points row uses LP_MIN/LP_MAX anchor so the 0–100 scale
#     is consistent with the logit probability axis
#
# Outputs:
#   outputs/figures/nomograms/nomogram_panels.png / .pdf
#   outputs/tables/nomogram_coefficients.csv
#   outputs/tables/nomogram_data_outcome1.csv
#   outputs/tables/nomogram_data_outcome2.csv
#
# TRIPOD-AI items covered:
#   - 13b: individual prediction tool
#   - 15 : coefficients and data saved for reproducibility
# =============================================================================

# =============================================================================
# CONSTANTS
# =============================================================================
PROB_TICKS = [0.01, 0.05, 0.10, 0.20, 0.30,
              0.40, 0.50, 0.60, 0.70, 0.80,
              0.90, 0.95, 0.99]

LP_MIN = np.log(0.01 / 0.99)   # −4.5951
LP_MAX = np.log(0.99 / 0.01)   # +4.5951
LP_RNG = LP_MAX - LP_MIN        #  9.1902

LEFT   = 20.0
RIGHT  = 105.0
W      = RIGHT - LEFT
LBL_X  = 18.5

MELD_TICKS = np.arange(0, 41, 5)
RAD_TICKS  = np.arange(-5, 6, 1)
TICK_H     = 0.25
FONTSIZE   = 10.5
FONTSIZE_S = 8.5


# =============================================================================
# STEP 1 — Build nomogram datasets
# =============================================================================

def make_nomo_dataset(df_sub, y_col, p_rad_oof,
                      outcome_col_name, csv_path):
    eps       = 1e-6
    p_clip    = np.clip(np.asarray(p_rad_oof, float), eps, 1 - eps)
    rad_score = np.log(p_clip / (1 - p_clip))
    df_nomo   = pd.DataFrame({
        outcome_col_name: df_sub[y_col].values.astype(int),
        "MELD":           df_sub[COL_MELD].values.astype(float),
        "RadScore":       rad_score,
    })
    df_nomo.to_csv(csv_path, index=False)
    print(f"  Dataset saved → {csv_path.name}  "
          f"n={len(df_nomo)}  RadScore [{rad_score.min():.2f}, {rad_score.max():.2f}]")
    return df_nomo


print("▶  Building nomogram datasets")
df_nomo_o1 = make_nomo_dataset(
    df_sub=df_o1, y_col=COL_HE_MILD, p_rad_oof=p_o1_rad,
    outcome_col_name="HE_Mild",
    csv_path=TAB_DIR / "nomogram_data_outcome1.csv",
)
df_nomo_o2 = make_nomo_dataset(
    df_sub=df_o2, y_col=COL_HE_MODSEV, p_rad_oof=p_o2_rad,
    outcome_col_name="HE_ModSev",
    csv_path=TAB_DIR / "nomogram_data_outcome2.csv",
)


# =============================================================================
# STEP 2 — Fit nomogram logistic models (unpenalized, full dataset)
# =============================================================================

def fit_nomo_model(df_nomo, outcome_col):
    X  = df_nomo[["MELD", "RadScore"]].values
    y  = df_nomo[outcome_col].values.astype(int)
    lr = LogisticRegression(penalty=None, solver="lbfgs", max_iter=2000)
    lr.fit(X, y)
    b0     = float(lr.intercept_[0])
    b_meld = float(lr.coef_[0][0])
    b_rad  = float(lr.coef_[0][1])
    print(f"  b0={b0:.4f}  b_meld={b_meld:.4f}  b_rad={b_rad:.4f}")
    return b0, b_meld, b_rad


print("\n▶  Fitting nomogram models")
b0_o1, b_meld_o1, b_rad_o1 = fit_nomo_model(df_nomo_o1, "HE_Mild")
b0_o2, b_meld_o2, b_rad_o2 = fit_nomo_model(df_nomo_o2, "HE_ModSev")

coef_df = pd.DataFrame([
    {"Outcome": "Outcome 1 — Mild vs No HE",
     "b0": b0_o1, "b_meld": b_meld_o1, "b_rad": b_rad_o1},
    {"Outcome": "Outcome 2 — Mod-Sev vs Mild HE",
     "b0": b0_o2, "b_meld": b_meld_o2, "b_rad": b_rad_o2},
])
coef_df.to_csv(TAB_DIR / "nomogram_coefficients.csv", index=False)
print(f"\n  ✓ Coefficients saved → nomogram_coefficients.csv")


# =============================================================================
# STEP 3 — Axis mapping helpers
# =============================================================================

def make_full_mapper(lo, hi):
    """Linear mapper: predictor value → x coordinate (full width LEFT→RIGHT)."""
    def mapper(value):
        return LEFT + (value - lo) / (hi - lo) * W
    return mapper


def pts_to_x(pts):
    """Points (0–100) → x coordinate."""
    return LEFT + pts / 100.0 * W


def prob_to_x(prob):
    """Probability → x coordinate (logit scale)."""
    lp = np.log(prob / (1 - prob))
    return LEFT + (lp - LP_MIN) / LP_RNG * W


# =============================================================================
# STEP 4 — Single-panel drawing function
# =============================================================================

def draw_panel(ax, b0, b_meld, b_rad, data_meld, data_rad, title):
    """
    Draw one nomogram panel on the given Axes object.

    Classic 5-row layout:
      Row 1 : Points       (0–100)
      Row 2 : MELD Score   (0–40, full width, KDE)
      Row 3 : RadScore     (−5/+5, full width, KDE)
      Row 4 : Total Points (0–100)
      Row 5 : Probability  (red, logit)
    """

    meld_mapper = make_full_mapper(0, 40)
    rad_mapper  = make_full_mapper(-5, 5)

    ax.set_xlim(0, 115)
    ax.set_ylim(0, 13)
    ax.axis("off")

    ROW = {
        "points":      11.5,
        "meld":         9.0,
        "rad":          6.5,
        "total":        4.0,
        "probability":  1.5,
    }

    # ------------------------------------------------------------------
    # Helpers local to this panel
    # ------------------------------------------------------------------

    def _draw_axis(row_y, row_label, ticks, mapper,
                   tick_fmt=None, color="black",
                   lfs=None, kde_data=None, kde_range=None):
        lfs = lfs or FONTSIZE
        ax.text(LBL_X, row_y, row_label,
                ha="right", va="center",
                fontsize=lfs, fontweight="bold", color=color)

        valid = [(t, mapper(t)) for t in ticks
                 if LEFT - 0.5 <= mapper(t) <= RIGHT + 0.5]
        if not valid:
            return

        ax.plot([valid[0][1], valid[-1][1]], [row_y, row_y],
                color=color, linewidth=1.8,
                solid_capstyle="butt", zorder=2)

        for t, x in valid:
            ax.plot([x, x], [row_y - TICK_H, row_y + TICK_H],
                    color=color, linewidth=1.5, zorder=3)
            lbl = (tick_fmt(t) if tick_fmt
                   else str(int(t)) if float(t) == int(t)
                   else f"{t:.1f}")
            ax.text(x, row_y - TICK_H - 0.18, lbl,
                    ha="center", va="top",
                    fontsize=FONTSIZE_S, color=color)

        # KDE cloud — aligned to mapper
        if kde_data is not None and len(kde_data) > 3:
            try:
                kde    = gaussian_kde(kde_data)
                lo_k   = kde_range[0] if kde_range else float(kde_data.min())
                hi_k   = kde_range[1] if kde_range else float(kde_data.max())
                x_eval = np.linspace(lo_k, hi_k, 150)
                x_plot = np.array([mapper(v) for v in x_eval])
                mask   = (x_plot >= LEFT) & (x_plot <= RIGHT)
                dens   = (kde(x_eval) / kde(x_eval).max()) * 0.38
                ax.fill_between(x_plot[mask], row_y,
                                row_y + dens[mask],
                                color="gray", alpha=0.13, zorder=1)
            except Exception:
                pass

    # ------------------------------------------------------------------
    # Vertical dashed grid
    # ------------------------------------------------------------------
    for v in range(0, 101, 10):
        xd = pts_to_x(v)
        ax.plot([xd, xd],
                [ROW["probability"] - 0.4, ROW["points"] + 0.3],
                color="#dddddd", lw=0.7, ls="--", alpha=0.5, zorder=0)

    # ------------------------------------------------------------------
    # Row 1: Points
    # ------------------------------------------------------------------
    _draw_axis(ROW["points"], "Points",
               np.arange(0, 101, 10), pts_to_x)

    # ------------------------------------------------------------------
    # Row 2: MELD Score
    # ------------------------------------------------------------------
    _draw_axis(ROW["meld"], "MELD Score",
               MELD_TICKS, meld_mapper,
               kde_data=data_meld, kde_range=(0, 40))

    # ------------------------------------------------------------------
    # Row 3: RadScore
    # ------------------------------------------------------------------
    _draw_axis(ROW["rad"], "RadScore",
               RAD_TICKS, rad_mapper,
               tick_fmt=lambda v: f"{int(v):+d}",
               lfs=9.5,
               kde_data=data_rad, kde_range=(-5, 5))

    # ------------------------------------------------------------------
    # Row 4: Total Points
    # ------------------------------------------------------------------
    _draw_axis(ROW["total"], "Total\nPoints",
               np.arange(0, 101, 10), pts_to_x,
               lfs=9.5)

    # ------------------------------------------------------------------
    # Row 5: Probability (red, logit scale)
    # ------------------------------------------------------------------
    prob_color = "#A60628"
    ax.text(LBL_X, ROW["probability"], "Probability\nof HE",
            ha="right", va="center",
            fontsize=FONTSIZE, fontweight="bold", color=prob_color)

    x_lo = prob_to_x(PROB_TICKS[0])
    x_hi = prob_to_x(PROB_TICKS[-1])
    ax.plot([x_lo, x_hi], [ROW["probability"], ROW["probability"]],
            color=prob_color, linewidth=1.8,
            solid_capstyle="butt", zorder=2)

    for prob in PROB_TICKS:
        x = prob_to_x(prob)
        ax.plot([x, x],
                [ROW["probability"] - TICK_H,
                 ROW["probability"] + TICK_H],
                color=prob_color, linewidth=1.5, zorder=3)
        lbl = (f"{int(prob*100)}%"
               if prob not in (0.01, 0.05, 0.95, 0.99)
               else f"{prob:.2f}")
        ax.text(x, ROW["probability"] - TICK_H - 0.18,
                lbl, ha="center", va="top",
                fontsize=FONTSIZE_S, fontweight="bold",
                color=prob_color)

    # ------------------------------------------------------------------
    # Panel title
    # ------------------------------------------------------------------
    ax.set_title(title, fontsize=11, fontweight="bold",
                 color="#222222", pad=8)

    # ------------------------------------------------------------------
    # Usage footnote
    # ------------------------------------------------------------------
    ax.text(
        (LEFT + RIGHT) / 2, 0.3,
        "Draw a vertical line from each predictor to the Points axis "
        "and sum the values. Project Total Points to the Probability axis.",
        ha="center", va="bottom",
        fontsize=6.5, color="#555555", style="italic",
        wrap=True,
    )


# =============================================================================
# STEP 5 — Render two-panel figure
# =============================================================================

print("\n▶  Rendering two-panel nomogram figure")

fig, axes = plt.subplots(
    1, 2,
    figsize=(22, 8.5),
    facecolor="white",
)
fig.patch.set_facecolor("white")

# Panel A — Outcome 1
draw_panel(
    ax        = axes[0],
    b0        = b0_o1,
    b_meld    = b_meld_o1,
    b_rad     = b_rad_o1,
    data_meld = df_nomo_o1["MELD"].values,
    data_rad  = df_nomo_o1["RadScore"].values,
    title     = ("A  —  Outcome 1\n"
                 "Prediction of Mild Hepatic Encephalopathy\n"
                 "(MELD Score + Radiomic Signature)"),
)

# Panel B — Outcome 2
draw_panel(
    ax        = axes[1],
    b0        = b0_o2,
    b_meld    = b_meld_o2,
    b_rad     = b_rad_o2,
    data_meld = df_nomo_o2["MELD"].values,
    data_rad  = df_nomo_o2["RadScore"].values,
    title     = ("B  —  Outcome 2\n"
                 "Differentiation of Moderate-to-Severe vs Mild HE\n"
                 "(MELD Score + Radiomic Signature)"),
)

plt.tight_layout(pad=3.0)

for ext in ("png", "pdf"):
    fig.savefig(NOM_DIR / f"nomogram_panels.{ext}",
                dpi=300, bbox_inches="tight",
                facecolor="white")

plt.show()
plt.close(fig)
print(f"  ✓ Saved → {(NOM_DIR / 'nomogram_panels')}.png / .pdf")


# =============================================================================
# STEP 6 — Also save individual panels for Supplementary
# =============================================================================

for outcome_label, b0, b_meld, b_rad, data_meld, data_rad, title, fname in [
    (
        "Outcome 1 — Mild vs No HE",
        b0_o1, b_meld_o1, b_rad_o1,
        df_nomo_o1["MELD"].values, df_nomo_o1["RadScore"].values,
        ("Nomogram — Outcome 1: Prediction of Mild Hepatic Encephalopathy\n"
         "(MELD Score + Radiomic Signature)"),
        "nomogram_outcome1",
    ),
    (
        "Outcome 2 — Mod-Sev vs Mild HE",
        b0_o2, b_meld_o2, b_rad_o2,
        df_nomo_o2["MELD"].values, df_nomo_o2["RadScore"].values,
        ("Nomogram — Outcome 2: Differentiation of Moderate-to-Severe "
         "vs Mild Hepatic Encephalopathy\n"
         "(MELD Score + Radiomic Signature)"),
        "nomogram_outcome2",
    ),
]:
    fig_s, ax_s = plt.subplots(figsize=(11, 8.5), facecolor="white")
    draw_panel(ax=ax_s, b0=b0, b_meld=b_meld, b_rad=b_rad,
               data_meld=data_meld, data_rad=data_rad, title=title)
    plt.tight_layout(pad=2.0)
    for ext in ("png", "pdf"):
        fig_s.savefig(NOM_DIR / f"{fname}.{ext}",
                      dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig_s)
    print(f"  ✓ Individual panel saved → {fname}.png / .pdf")


# =============================================================================
# STEP 7 — Inline HTML legend
# =============================================================================

def print_nomogram_legend(outcome_label, fname, b0, b_meld, b_rad,
                          df_nomo, outcome_col):
    X   = df_nomo[["MELD", "RadScore"]].values
    y   = df_nomo[outcome_col].values.astype(int)
    lr  = LogisticRegression(penalty=None, solver="lbfgs", max_iter=2000)
    lr.fit(X, y)
    auc_app = roc_auc_score(y, lr.predict_proba(X)[:, 1])

    th   = ("padding:5px 12px; text-align:center; font-weight:normal; "
            "border-top:2px solid #111; border-bottom:1px solid #111; "
            "background:#f7f7f7;")
    th_l = th.replace("center", "left")
    td   = "padding:5px 12px; text-align:center;"
    td_l = td.replace("center", "left")

    html = f"""
    <div style="font-family:'Helvetica Neue',Arial,sans-serif;
                font-size:12px; max-width:820px;
                margin-bottom:28px; margin-top:8px;">
      <p style="font-size:12px; margin:0 0 6px 0;">
        <b>Figure legend — Nomogram ({outcome_label})</b>
      </p>
      <p style="margin:0 0 10px 0; color:#333; line-height:1.6;">
        Clinical-radiomic nomogram integrating the MELD score and the
        MRI-derived Radiomic Signature for the estimation of individual
        probability of {outcome_label.lower()}.
        The Radiomic Score (RadScore) is derived from out-of-fold
        cross-validation predictions (logit-transformed) to ensure
        calibration integrity and avoid data leakage (TRIPOD-AI item 10a).
        Both predictor axes span the full figure width; the Points axis
        reflects the relative contribution of each predictor to the
        linear predictor (Iasonos et al.,
        <i>J Clin Oncol</i> 2008;26:1364–1370).
        Gray density clouds represent the empirical distribution of
        predictor values in the cohort (Gaussian KDE).
        The probability axis (red) is rendered on a logit scale
        anchored at 0.01–0.99.
        Apparent AUC on full data: <b>{auc_app:.3f}</b>
        (optimistic — validated estimates in Tables 2–3).
      </p>
      <p style="margin:0 0 6px 0;"><b>Model coefficients</b></p>
      <table style="border-collapse:collapse; width:auto; margin-bottom:12px;">
        <thead>
          <tr>
            <th style="{th_l}">Parameter</th>
            <th style="{th}">Value</th>
            <th style="{th}">Interpretation</th>
          </tr>
        </thead>
        <tbody>
          <tr style="border-bottom:0.5px solid #e8e8e8;">
            <td style="{td_l}">Intercept (b₀)</td>
            <td style="{td}">{b0:.4f}</td>
            <td style="{td}">Baseline log-odds</td>
          </tr>
          <tr style="border-bottom:0.5px solid #e8e8e8;">
            <td style="{td_l}">b_MELD</td>
            <td style="{td}">{b_meld:.4f}</td>
            <td style="{td}">Log-odds change per 1-unit MELD increase</td>
          </tr>
          <tr style="border-bottom:0.5px solid #e8e8e8;">
            <td style="{td_l}">b_RadScore</td>
            <td style="{td}">{b_rad:.4f}</td>
            <td style="{td}">Log-odds change per 1-unit RadScore increase</td>
          </tr>
          <tr style="border-top:2px solid #111;">
            <td style="{td_l}">Apparent AUC</td>
            <td style="{td}">{auc_app:.3f}</td>
            <td style="{td}">Full-data fit — optimistic (see Tables 2–3)</td>
          </tr>
        </tbody>
      </table>
      <p style="margin:0 0 6px 0;"><b>How to use</b></p>
      <ol style="color:#333; line-height:1.8; margin:0 0 8px 0;">
        <li>Locate the patient&#x2019;s MELD score on the MELD Score axis
            and draw a vertical line to the Points axis. Note the value.</li>
        <li>Locate the patient&#x2019;s RadScore on the RadScore axis
            and draw a vertical line to the Points axis. Note the value.</li>
        <li>Sum the two Points values → Total Points.</li>
        <li>Project Total Points down to the Probability axis to read
            the estimated individual probability of {outcome_label.lower()}.</li>
      </ol>
      <p style="font-size:11px; color:#555; margin:4px 0 0 0;">
        Saved: <code>{fname}.png / .pdf</code>
      </p>
    </div>"""
    display(HTML(html))


print_nomogram_legend(
    outcome_label="Outcome 1 — Mild vs No HE",
    fname="nomogram_outcome1",
    b0=b0_o1, b_meld=b_meld_o1, b_rad=b_rad_o1,
    df_nomo=df_nomo_o1, outcome_col="HE_Mild",
)
print_nomogram_legend(
    outcome_label="Outcome 2 — Mod-Sev vs Mild HE",
    fname="nomogram_outcome2",
    b0=b0_o2, b_meld=b_meld_o2, b_rad=b_rad_o2,
    df_nomo=df_nomo_o2, outcome_col="HE_ModSev",
)

print(f"\n✅  Cell 11 — Nomograms complete.")

In [ ]:
# =============================================================================
# Cell 12 — Reproducibility check and session info
# =============================================================================
# Final cell — verifies that all expected output files were generated,
# prints a full summary of results for quick inspection, and logs the
# computational environment for reproducibility.
#
# Steps:
#   1. Output file verification  : checks existence of all expected
#                                  figures, tables, and model outputs.
#   2. Performance summary       : reprints AUC, Brier, calibration,
#                                  and DeLong results from df_metrics.
#   3. Cross-system summary      : reprints AUC and permutation test
#                                  results from df_cs.
#   4. Nomogram coefficients     : reprints b0, b_meld, b_rad for
#                                  both outcomes.
#   5. Reproducibility parameters: logs all stochastic parameters
#                                  (random state, CV folds, bootstrap n,
#                                  C grid) for independent replication.
#   6. Session info              : Python version, OS, and key package
#                                  versions for environment logging.
#   7. Sharing checklist         : reminds the user of files to include
#                                  in the GitHub / Zenodo repository.
#
# TRIPOD-AI items covered:
#   - 15: reproducibility — code, environment, and outputs documented
# =============================================================================

import sys
import platform
import importlib
from datetime import datetime

# =============================================================================
# STEP 1 — Output file verification
# =============================================================================
expected_files = [
    # Tables
    TAB_DIR / "radiomic_HE_dataset_clean.csv",
    TAB_DIR / "oof_predictions_outcome1.csv",
    TAB_DIR / "oof_predictions_outcome2.csv",
    TAB_DIR / "performance_metrics.csv",
    TAB_DIR / "cross_system_validation.csv",
    TAB_DIR / "nomogram_coefficients.csv",
    TAB_DIR / "nomogram_data_outcome1.csv",
    TAB_DIR / "nomogram_data_outcome2.csv",
    TAB_DIR / "shap_features_outcome1.csv",
    TAB_DIR / "shap_features_outcome2.csv",
    # Calibration
    CAL_DIR / "calibration_outcome1.png",
    CAL_DIR / "calibration_outcome2.png",
    # DCA
    DCA_DIR / "dca_outcome1.png",
    DCA_DIR / "dca_outcome2.png",
    # SHAP
    SHA_DIR / "shap_beeswarm_outcome1.png",
    SHA_DIR / "shap_beeswarm_outcome2.png",
    SHA_DIR / "shap_bar_outcome1.png",
    SHA_DIR / "shap_bar_outcome2.png",
    # Bar chart
    BAR_DIR / "auc_barchart.png",
    # Cross-system
    CRS_DIR / "crosssystem_outcome1.png",
    CRS_DIR / "crosssystem_outcome2.png",
    # Nomograms
    NOM_DIR / "nomogram_outcome1.png",
    NOM_DIR / "nomogram_outcome2.png",
]

SEP = "=" * 65

print(SEP)
print("STEP 1 — OUTPUT FILE VERIFICATION")
print(SEP)

all_ok    = True
missing_f = []

for f in expected_files:
    exists = f.exists()
    status = "✅" if exists else "❌"
    print(f"  {status}  {f.relative_to(ROOT)}")
    if not exists:
        all_ok = False
        missing_f.append(f)

if all_ok:
    print(f"\n  ✅  All {len(expected_files)} output files present.")
else:
    print(f"\n  ⚠️   {len(missing_f)} file(s) missing — "
          f"re-run the corresponding cell(s):")
    for f in missing_f:
        print(f"       → {f.relative_to(ROOT)}")

# =============================================================================
# STEP 2 — Performance summary (from df_metrics, Cell 5)
# =============================================================================
print(f"\n{SEP}")
print("STEP 2 — PERFORMANCE SUMMARY (OOF, nested 5-fold CV)")
print(SEP)

for outcome in df_metrics["Outcome"].unique():
    print(f"\n  {outcome}")
    print(f"  {'Model':<12} {'AUC':>6}  {'95% CI':<17} "
          f"{'Brier':>6}  {'Slope':>6}  {'Intercept':>9}  "
          f"{'DeLong p':>10}  Sig.")
    print("  " + "-" * 72)
    df_out = df_metrics[df_metrics["Outcome"] == outcome]
    for _, r in df_out.iterrows():
        p_str = (
            "        —" if (isinstance(r.DeLong_p, float)
                            and np.isnan(r.DeLong_p))
            else "< 0.001  " if r.DeLong_p < 0.001
            else "< 0.01   " if r.DeLong_p < 0.01
            else f"  {r.DeLong_p:.3f}  "
        )
        print(
            f"  {r.Model:<12} "
            f"{r.AUC:>6.3f}  "
            f"({r.AUC_CI_lo:.3f}–{r.AUC_CI_hi:.3f})  "
            f"{r.Brier:>6.3f}  "
            f"{r.Cal_Slope:>6.3f}  "
            f"{r.Cal_Intercept:>9.3f}  "
            f"{p_str:>10}  "
            f"{str(r.Sig):>4}"
        )

# =============================================================================
# STEP 3 — Cross-system summary (from df_cs, Cell 10)
# =============================================================================
print(f"\n{SEP}")
print("STEP 3 — CROSS-SYSTEM VALIDATION SUMMARY")
print(SEP)

for outcome in df_cs["Outcome"].unique():
    print(f"\n  {outcome}")
    for path in df_cs["Path"].unique():
        df_p = df_cs[
            (df_cs["Outcome"] == outcome) &
            (df_cs["Path"]    == path)
        ]
        if df_p.empty:
            continue
        print(f"\n    {path}  "
              f"(train n={int(df_p['N_train'].iloc[0])}, "
              f"test n={int(df_p['N_test'].iloc[0])})")
        print(f"    {'Model':<12} {'AUC':>6}  {'95% CI':<17} "
              f"{'Brier':>6}  {'Perm p':>8}  Sig.")
        print("    " + "-" * 58)
        for _, r in df_p.iterrows():
            p_str = (
                "< 0.001" if r.Perm_p < 0.001
                else "< 0.01 " if r.Perm_p < 0.01
                else f"  {r.Perm_p:.3f}"
            )
            print(
                f"    {r.Model:<12} "
                f"{r.AUC:>6.3f}  "
                f"({r.AUC_CI_lo:.3f}–{r.AUC_CI_hi:.3f})  "
                f"{r.Brier:>6.3f}  "
                f"{p_str:>8}  "
                f"{r.Sig_perm:>4}"
            )

# =============================================================================
# STEP 4 — Nomogram coefficients
# =============================================================================
print(f"\n{SEP}")
print("STEP 4 — NOMOGRAM COEFFICIENTS")
print(SEP)

coef_check = pd.read_csv(TAB_DIR / "nomogram_coefficients.csv")
for _, r in coef_check.iterrows():
    print(f"\n  {r['Outcome']}")
    print(f"    Intercept (b0) : {r['b0']:+.4f}")
    print(f"    b_MELD         : {r['b_meld']:+.4f}")
    print(f"    b_RadScore     : {r['b_rad']:+.4f}")

# =============================================================================
# STEP 5 — Reproducibility parameters
# =============================================================================
print(f"\n{SEP}")
print("STEP 5 — REPRODUCIBILITY PARAMETERS")
print(SEP)
print(f"  Random state      : {RANDOM_STATE}")
print(f"  Bootstrap seed    : {BOOT_SEED}")
print(f"  CV folds          : {CV_FOLDS}")
print(f"  Bootstrap n       : {N_BOOT}")
print(f"  Permutation n     : 2,000")
print(f"  C grid            : {C_GRID['clf__C'].round(5).tolist()}")
print(f"  Penalty           : L1 (liblinear solver)")
print(f"  Class weight      : balanced")
print(f"  Calibration bins  : {N_BINS} (equal-frequency quantile)")
print(f"  SHAP top features : {TOP_N_SHAP}")
print(f"  Clean dataset     : {(TAB_DIR / 'radiomic_HE_dataset_clean.csv').resolve()}")
print(f"  OOF predictions   : {(TAB_DIR / 'oof_predictions_outcome1.csv').resolve()}")

# =============================================================================
# STEP 6 — Session info
# =============================================================================
print(f"\n{SEP}")
print("STEP 6 — SESSION INFO")
print(SEP)
print(f"  Date / time  : {datetime.now().strftime('%Y-%m-%d  %H:%M:%S')}")
print(f"  Platform     : {platform.platform()}")
print(f"  Python       : {sys.version.split()[0]}")

packages = [
    "numpy", "pandas", "sklearn",
    "scipy", "matplotlib", "shap",
    "statsmodels",
]
for pkg in packages:
    alias = {"sklearn": "scikit-learn"}.get(pkg, pkg)
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "unknown")
    except ImportError:
        ver = "not installed"
    print(f"  {alias:<16} : {ver}")

# =============================================================================
# STEP 7 — Sharing checklist (GitHub / Zenodo)
# =============================================================================
print(f"\n{SEP}")
print("STEP 7 — SHARING CHECKLIST  (GitHub / Zenodo)")
print(SEP)

checklist = [
    ("HE_radiomics_pipeline.ipynb",              "this notebook"),
    ("data/radiomic_HE_dataset_clean.csv",        "clean dataset"),
    ("outputs/tables/performance_metrics.csv",    "Table 2 / Table 3"),
    ("outputs/tables/cross_system_validation.csv","Supplementary Table ST3"),
    ("outputs/tables/nomogram_coefficients.csv",  "nomogram model"),
    ("outputs/tables/oof_predictions_outcome*.csv","OOF predictions"),
    ("outputs/tables/shap_features_outcome*.csv", "SHAP feature ranking"),
    ("outputs/figures/",                          "all figures (PNG/PDF)"),
    ("requirements.txt",                          "package versions"),
    ("README.md",                                 "usage instructions"),
    ("LICENSE",                                   "MIT license"),
]
for item, desc in checklist:
    print(f"  ☐  {item:<48}  ← {desc}")

# =============================================================================
# FINAL BANNER
# =============================================================================
print(f"\n{SEP}")
if all_ok:
    print("  ✅  PIPELINE COMPLETE — all outputs verified.")
else:
    print(f"  ⚠️   PIPELINE COMPLETE — {len(missing_f)} file(s) missing.")
print(f"  Notebook : HE_radiomics_pipeline.ipynb")
print(f"  Outputs  : {OUT_DIR.resolve()}")
print(SEP)